# AQUA — Data Quality Audit

## 1. Research Project

**Project title:** Statistical Analysis and Predictive Modelling of Aquaculture Water Quality

**Research objective:**  
To investigate water-quality measurements from Indian aquaculture farms, assess the reliability and completeness of the available data, identify data-quality limitations, and establish a defensible foundation for subsequent statistical analysis and predictive modelling.

## 2. Dataset Overview

**Dataset:** `water_quality.csv`  
**Source:** Fish Welfare Initiative (FWI) — Farm Data India  
**Original source:** https://github.com/fish-welfare-initiative/fwi-farm-data-india

The dataset contains water-quality observations collected from aquaculture farms in the Eluru and Nellore regions of India.

The downloaded dataset contains:

- 12,431 records
- 53 columns
- 317 unique pond identifiers
- Observation dates from 19 June 2021 to 6 August 2026

## 3. Purpose of This Audit

This notebook examines the dataset before any statistical analysis or model development.

The audit will:

1. Verify the dataset structure and column definitions.
2. Examine missing values and their distribution.
3. Compare measurement availability across morning and evening observations.
4. Identify duplicate records and potential inconsistencies.
5. Investigate measurement protocols and changes over time.
6. Identify potential target leakage and variables unsuitable for predictive modelling.
7. Document data limitations and decisions for subsequent analysis.

## 4. Methodological Considerations

The dataset documentation identifies several important considerations:

- Some parameters, including alkalinity and hardness, are intentionally uncollected.
- Missing values may represent measurements that were not collected rather than data-entry failures.
- Water-quality parameters have different measurement frequencies.
- Ammonia measurement protocols changed in August 2025. The historical TAN and newer NH3 measurements must not be treated as directly interchangeable.
- Measurement equipment and observers may introduce variation.
- The `Parameters out of range` field is closely associated with the water-quality classification and must be investigated before selecting predictive-model features.
- Nellore observations identified as retracted in the source documentation must be handled separately and excluded where applicable.

No missing values will be automatically imputed or records deleted without a documented methodological justification.

## 5. Audit Principle

The original downloaded dataset will remain unchanged.

All cleaning decisions, exclusions, transformations and derived datasets will be documented and saved separately. Conclusions will be based on observed data and verified source documentation rather than assumptions.

## 6. Expected Outcome

At the end of this notebook, we aim to establish:

- The structural integrity of the dataset.
- The extent and patterns of missing data.
- Potential measurement and recording limitations.
- Variables suitable for further analysis.
- A documented data-preparation strategy for the next research stage.

In [1]:
# ============================================================
# AQUA — DATA QUALITY AUDIT
# Cell 1: Environment, Dataset Loading and Initial Verification
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "water_quality.csv"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

# Verify dataset location
assert RAW_DATA_PATH.exists(), (
    f"Dataset not found: {RAW_DATA_PATH}"
)

# Load original dataset without modifying it
df = pd.read_csv(RAW_DATA_PATH)

# Initial structural checks
assert df.shape == (12431, 53), (
    f"Unexpected dataset dimensions: {df.shape}"
)

assert df.columns.is_unique, "Duplicate column names detected."

assert df["pond_id"].notna().all(), "Missing pond identifiers detected."

# Parse dates for verification only
collection_dates = pd.to_datetime(
    df["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

# Initial verification report
print("=" * 65)
print("AQUA — INITIAL DATASET VERIFICATION")
print("=" * 65)

print(f"\nDataset dimensions : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique ponds       : {df['pond_id'].nunique():,}")
print(f"Regions            : {df['region'].nunique()}")
print(f"Duplicate rows     : {df.duplicated().sum():,}")
print(f"Missing pond IDs   : {df['pond_id'].isna().sum():,}")

print("\nDATE VERIFICATION")
print(f"Unparseable dates  : {collection_dates.isna().sum():,}")
print(f"Earliest date      : {collection_dates.min().date()}")
print(f"Latest date        : {collection_dates.max().date()}")

print("\nREGIONAL DISTRIBUTION")
print(df["region"].value_counts(dropna=False).to_string())

print("\nMEASUREMENT TYPE DISTRIBUTION")
print(df["Type"].value_counts(dropna=False).to_string())

print("\nTARGET DISTRIBUTION")
print(df["Is WQ in range?"].value_counts(dropna=False).to_string())

# Confirm date parsing
assert collection_dates.notna().all(), "Some dates could not be parsed."

print("\n" + "=" * 65)
print("INITIAL VERIFICATION COMPLETED SUCCESSFULLY")
print("=" * 65)

AQUA — INITIAL DATASET VERIFICATION

Dataset dimensions : 12,431 rows × 53 columns
Unique ponds       : 317
Regions            : 2
Duplicate rows     : 0
Missing pond IDs   : 0

DATE VERIFICATION
Unparseable dates  : 0
Earliest date      : 2021-06-19
Latest date        : 2026-08-06

REGIONAL DISTRIBUTION
region
Eluru      9248
Nellore    3183

MEASUREMENT TYPE DISTRIBUTION
Type
Morning    6454
Evening    5975
NaN           2

TARGET DISTRIBUTION
Is WQ in range?
Yes    10064
No      2367

INITIAL VERIFICATION COMPLETED SUCCESSFULLY


In [2]:
# ============================================================
# Cell 2: Measurement Availability by Observation Type
# ============================================================

parameters = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Temp (in °C)",
    "TDS (ppt)"
]

print("=" * 70)
print("RECORDS BY MEASUREMENT TYPE")
print("=" * 70)

print(df["Type"].value_counts(dropna=False).to_string())

print("\n" + "=" * 70)
print("MEASUREMENT AVAILABILITY BY TYPE")
print("=" * 70)

for parameter in parameters:

    availability = (
        df.groupby("Type", dropna=False)[parameter]
        .agg(
            total_records="size",
            available="count",
            missing=lambda x: x.isna().sum()
        )
    )

    availability["missing_percentage"] = (
        availability["missing"]
        / availability["total_records"]
        * 100
    ).round(2)

    print(f"\n{parameter}")
    print(availability.to_string())

print("\n" + "=" * 70)
print("MEASUREMENT AVAILABILITY ANALYSIS COMPLETED")
print("=" * 70)

RECORDS BY MEASUREMENT TYPE
Type
Morning    6454
Evening    5975
NaN           2

MEASUREMENT AVAILABILITY BY TYPE

DO (mg/L)
         total_records  available  missing  missing_percentage
Type                                                          
Evening           5975       5970        5                0.08
Morning           6454       6440       14                0.22
NaN                  2          2        0                0.00

pH
         total_records  available  missing  missing_percentage
Type                                                          
Evening           5975       5970        5                0.08
Morning           6454       6440       14                0.22
NaN                  2          2        0                0.00

Turbidity (in cm)
         total_records  available  missing  missing_percentage
Type                                                          
Evening           5975       1360     4615               77.24
Morning           6454       353

In [3]:
# ============================================================
# Cell 3: Investigate Records with Missing Observation Type
# ============================================================

missing_type_records = df[df["Type"].isna()]

inspection_columns = [
    "Date of data collection",
    "Time of data collection",
    "pond_id",
    "region",
    "Type",
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "Temp (in °C)",
    "TDS (ppt)",
    "Is WQ in range?"
]

print("=" * 75)
print("RECORDS WITH MISSING OBSERVATION TYPE")
print("=" * 75)

print(f"\nNumber of records: {len(missing_type_records)}")

display(
    missing_type_records[inspection_columns].to_string(index=False)
)

print("\n" + "=" * 75)
print("MISSING TYPE INVESTIGATION COMPLETED")
print("=" * 75)

RECORDS WITH MISSING OBSERVATION TYPE

Number of records: 2


'Date of data collection Time of data collection       pond_id  region Type  DO (mg/L)   pH  Turbidity (in cm)  Temp (in °C)  TDS (ppt) Is WQ in range?\n             07/14/2021                   00:00 ara2_f1a474ab Nellore  NaN       6.20 8.40              19.00         29.20        NaN              No\n             07/14/2021                   00:00 ara2_17b9d7b4 Nellore  NaN       8.50 8.63              31.00         28.60        NaN              No'


MISSING TYPE INVESTIGATION COMPLETED


In [4]:
# ============================================================
# Cell 4: Duplicate Measurement Keys and Target Leakage Audit
# ============================================================

print("=" * 75)
print("PART 1: DUPLICATE MEASUREMENT KEY INVESTIGATION")
print("=" * 75)

# Check repeated pond-date-time combinations
measurement_key = [
    "Date of data collection",
    "Time of data collection",
    "pond_id"
]

duplicate_measurements = df[
    df.duplicated(subset=measurement_key, keep=False)
].sort_values(measurement_key)

print("\nRepeated pond-date-time combinations:")
print(f"Records involved: {len(duplicate_measurements)}")

if duplicate_measurements.empty:
    print("No repeated pond-date-time combinations found.")
else:
    display(duplicate_measurements[measurement_key + ["Type"]])

# Check whether including Type changes the result
measurement_key_with_type = measurement_key + ["Type"]

duplicate_measurements_with_type = df[
    df.duplicated(
        subset=measurement_key_with_type,
        keep=False
    )
].sort_values(measurement_key_with_type)

print("\nRepeated pond-date-time-Type combinations:")
print(f"Records involved: {len(duplicate_measurements_with_type)}")

if duplicate_measurements_with_type.empty:
    print("No repeated combinations found.")
else:
    display(
        duplicate_measurements_with_type[
            measurement_key_with_type
        ]
    )


print("\n" + "=" * 75)
print("PART 2: TARGET LEAKAGE INVESTIGATION")
print("=" * 75)

target = "Is WQ in range?"
potential_leakage = "Parameters out of range"

print("\nTarget distribution:")
print(df[target].value_counts(dropna=False).to_string())

print("\nPotential leakage column distribution:")
print(
    df[potential_leakage]
    .value_counts(dropna=False)
    .to_string()
)

print("\nTarget versus potential leakage cross-tabulation:")

leakage_crosstab = pd.crosstab(
    df[target],
    df[potential_leakage],
    margins=True,
    dropna=False
)

display(leakage_crosstab)

print("\nRecords with missing potential leakage values:")
print(df[potential_leakage].isna().sum())

print("\n" + "=" * 75)
print("DUPLICATE KEY AND TARGET LEAKAGE AUDIT COMPLETED")
print("=" * 75)

PART 1: DUPLICATE MEASUREMENT KEY INVESTIGATION

Repeated pond-date-time combinations:
Records involved: 4


,Date of data collection,Time of data collection,pond_id,Type
12429,06/19/2021,00:00,ara2_407f70f7,Evening
12430,06/19/2021,00:00,ara2_407f70f7,Morning
12424,07/06/2021,00:00,ara2_5a3d15dd,Evening
12426,07/06/2021,00:00,ara2_5a3d15dd,Morning



Repeated pond-date-time-Type combinations:
Records involved: 0
No repeated combinations found.

PART 2: TARGET LEAKAGE INVESTIGATION

Target distribution:
Is WQ in range?
Yes    10064
No      2367

Potential leakage column distribution:
Parameters out of range
NaN                                              10064
Dissolved Oxygen                                  1078
Ph                                                 403
Turbidity                                          302
Dissolved Oxygen, Ph                               293
Ammonia Tan Nh3N                                    52
Dissolved Oxygen, Turbidity                         52
Dissolved Oxygen, Ammonia Tan Nh3N                  47
Dissolved Oxygen, Ph, Ammonia Tan Nh3N              34
Ph, Ammonia Tan Nh3N                                30
Dissolved Oxygen, Ammonia Nh3                       23
Ph, Ammonia Nh3                                     15
Dissolved Oxygen, Ph, Ammonia Nh3                   13
Ammonia Nh3            

Parameters out of range,Ammonia Nh3,Ammonia Tan Nh3N,Dissolved Oxygen,"Dissolved Oxygen, Ammonia Nh3","Dissolved Oxygen, Ammonia Tan Nh3N","Dissolved Oxygen, Ph","Dissolved Oxygen, Ph, Ammonia Nh3","Dissolved Oxygen, Ph, Ammonia Tan Nh3N","Dissolved Oxygen, Ph, Turbidity","Dissolved Oxygen, Turbidity","Dissolved Oxygen, Turbidity, Ammonia Tan Nh3N",Ph,"Ph, Ammonia Nh3","Ph, Ammonia Tan Nh3N","Ph, Turbidity",Turbidity,"Turbidity, Ammonia Tan Nh3N",NaN,All
Is WQ in range?,,,,,,,,,,,,,,,,,,,
No,11,52,1078,23,47,293,13,34,4,52,1,403,15,30,5,302,4,0,2367
Yes,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10064,10064
All,11,52,1078,23,47,293,13,34,4,52,1,403,15,30,5,302,4,10064,12431



Records with missing potential leakage values:
10064

DUPLICATE KEY AND TARGET LEAKAGE AUDIT COMPLETED


In [5]:
# AQUA: Audit column cardinality and constant values

import pandas as pd

audit_rows = []

for column in df.columns:
    non_missing = df[column].dropna()
    total_non_missing = len(non_missing)
    unique_values = non_missing.nunique()

    if total_non_missing == 0:
        most_common_value = None
        most_common_count = 0
        most_common_percentage = None

    else:
        value_counts = non_missing.value_counts(dropna=True)
        most_common_value = value_counts.index[0]
        most_common_count = int(value_counts.iloc[0])
        most_common_percentage = (
            most_common_count / total_non_missing
        ) * 100

    audit_rows.append({
        "Column": column,
        "Non-missing values": total_non_missing,
        "Unique values": unique_values,
        "Most common value": most_common_value,
        "Most common count": most_common_count,
        "Most common (%)": (
            round(most_common_percentage, 2)
            if most_common_percentage is not None
            else None
        )
    })

cardinality_report = pd.DataFrame(audit_rows)

print("COLUMN CARDINALITY REPORT")
display(cardinality_report.to_string(index=False))

print("\nCONSTANT COLUMNS")
constant_columns = cardinality_report[
    (cardinality_report["Non-missing values"] > 0) &
    (cardinality_report["Unique values"] == 1)
]
display(constant_columns.to_string(index=False))

print("\nNEAR-CONSTANT COLUMNS (99.5% OR MORE)")
near_constant_columns = cardinality_report[
    (cardinality_report["Non-missing values"] > 0) &
    (cardinality_report["Unique values"] > 1) &
    (cardinality_report["Most common (%)"] >= 99.5)
]
display(near_constant_columns.to_string(index=False))

print("\nCOMPLETELY EMPTY COLUMNS")
empty_columns = cardinality_report[
    cardinality_report["Non-missing values"] == 0
]
display(empty_columns.to_string(index=False))

COLUMN CARDINALITY REPORT


'                                                       Column  Non-missing values  Unique values                                      Most common value  Most common count  Most common (%)\n                                                      pond_id               12431            317                                          ara2_09faa72c                170             1.37\n                                                       region               12431              2                                                  Eluru               9248            74.39\n                                      Date of data collection               12431           1210                                             05/20/2026                 30             0.24\n                                      Time of data collection               12431            356                                                  07:10                216             1.74\n                                                      


CONSTANT COLUMNS


'Empty DataFrame\nColumns: [Column, Non-missing values, Unique values, Most common value, Most common count, Most common (%)]\nIndex: []'


NEAR-CONSTANT COLUMNS (99.5% OR MORE)


'               Column  Non-missing values  Unique values Most common value  Most common count  Most common (%)\nHave fish been helped               12431              2                No              12374            99.54'


COMPLETELY EMPTY COLUMNS


'           Column  Non-missing values  Unique values Most common value  Most common count  Most common (%)\nAlkalinity (mg/L)                   0              0              None                  0              NaN\n  Hardness (mg/L)                   0              0              None                  0              NaN'

In [6]:
# AQUA: Classify columns by analytical role and preliminary predictor eligibility

column_roles = {
    # Identifiers and visit metadata
    "pond_id": (
        "Identifier",
        "Exclude from direct prediction",
        "Identifies repeated observations from the same pond."
    ),
    "region": (
        "Geographic metadata",
        "Review",
        "May capture regional differences; assess generalisation risk."
    ),
    "Date of data collection": (
        "Visit metadata",
        "Review",
        "May capture temporal trends and changes in measurement protocols."
    ),
    "Time of data collection": (
        "Visit metadata",
        "Review",
        "Potentially useful for measurement context."
    ),
    "Type": (
        "Visit metadata",
        "Review",
        "Morning/evening context is important for dissolved oxygen."
    ),
    "Is follow up": (
        "Visit process",
        "Exclude from primary model",
        "Follow-up visits are triggered by previous water-quality problems."
    ),
    "Is follow up possible": (
        "Follow-up process",
        "Exclude",
        "Relates to follow-up scheduling rather than the initial measurement."
    ),
    "Reason follow up not possible": (
        "Follow-up process",
        "Exclude",
        "Administrative information about follow-up scheduling."
    ),
    "Group": (
        "Administrative metadata",
        "Review",
        "Internal farm grouping; may encode programme or study participation."
    ),
    "Pond status": (
        "Study/intervention metadata",
        "Review",
        "Treatment/control/assessment status may affect interpretation."
    ),
    "Observer": (
        "Measurement metadata",
        "Review",
        "Observer differences may introduce measurement variation."
    ),
    "Equipment": (
        "Measurement metadata",
        "Review",
        "Different equipment may produce measurement variation."
    ),
    "Weather": (
        "Environmental context",
        "Review",
        "Potential contextual predictor if available at prediction time."
    ),

    # Water-quality measurements
    "DO (mg/L)": (
        "Water-quality measurement",
        "Candidate",
        "Core dissolved oxygen measurement."
    ),
    "pH": (
        "Water-quality measurement",
        "Candidate",
        "Core pH measurement."
    ),
    "Turbidity (in cm)": (
        "Water-quality measurement",
        "Candidate with limitations",
        "Substantial missingness and morning-only collection."
    ),
    "Ammonia—TAN (NH3-N) (mg/L)": (
        "Water-quality measurement",
        "Candidate with protocol limitation",
        "Old TAN method used until August 2025."
    ),
    "Ammonia—TAN (NH3) (mg/L)": (
        "Water-quality measurement",
        "Candidate with protocol limitation",
        "Transition-period ammonia measurement."
    ),
    "Ammonia—NH3 (mg/L)": (
        "Water-quality measurement",
        "Candidate with protocol limitation",
        "New primary ammonia measurement from August 2025."
    ),
    "Temp (in °C)": (
        "Water-quality measurement",
        "Candidate",
        "Core water-temperature measurement."
    ),
    "TDS (ppt)": (
        "Water-quality measurement",
        "Candidate with limitations",
        "Substantial missingness."
    ),
    "Alkalinity (mg/L)": (
        "Water-quality measurement",
        "Unavailable",
        "No recorded values in this dataset."
    ),
    "Hardness (mg/L)": (
        "Water-quality measurement",
        "Unavailable",
        "No recorded values in this dataset."
    ),
    "Water color": (
        "Water-quality observation",
        "Candidate with limitations",
        "Visual categorical observation."
    ),

    # Target and target-derived information
    "Is WQ in range?": (
        "Target",
        "Target only",
        "Outcome to be predicted; never use as an input feature."
    ),
    "Parameters out of range": (
        "Target-derived information",
        "Exclude",
        "Directly reveals the target outcome."
    ),

    # FWI-prescribed corrective actions
    "Corrective actions requested": (
        "Corrective action",
        "Exclude",
        "Action recommended after water-quality assessment."
    ),
    "Corrective actions requested (other)": (
        "Corrective action",
        "Exclude",
        "Free-text recommendation."
    ),
    "Corrective actions amount requested": (
        "Corrective action",
        "Exclude",
        "Recommended action dosage."
    ),
    "Corrective actions": (
        "Corrective action",
        "Exclude",
        "Summary of corrective actions."
    ),
    "Corrective actions implemented": (
        "Corrective action",
        "Exclude",
        "Records farmer implementation of recommended actions."
    ),
    "Corrective actions implementation date": (
        "Corrective action",
        "Exclude",
        "Records when recommended actions were implemented."
    ),
    "Corrective actions taken": (
        "Corrective action",
        "Exclude",
        "Records actions taken after recommendations."
    ),
    "Corrective actions taken (other)": (
        "Corrective action",
        "Exclude",
        "Free-text description of actions taken."
    ),
    "Corrective actions taken (details)": (
        "Corrective action",
        "Exclude",
        "Additional details about actions taken."
    ),
    "Non-prescribed corrective actions taken": (
        "Corrective action",
        "Exclude",
        "Records additional interventions."
    ),
    "Reason not implemented": (
        "Corrective action",
        "Exclude",
        "Explains why recommended actions were not implemented."
    ),
    "Water quality improved after corrective actions": (
        "Post-intervention outcome",
        "Exclude",
        "Records improvement after corrective actions."
    ),
    "Corrective action notes": (
        "Corrective action",
        "Exclude",
        "Notes about corrective actions."
    ),

    # Self-initiated corrective actions
    "Self-initiated corrective actions taken": (
        "Self-initiated intervention",
        "Exclude",
        "Records actions taken independently by farmers."
    ),
    "Self-initiated corrective actions implemented on (exact date)": (
        "Self-initiated intervention",
        "Exclude",
        "Records intervention date."
    ),
    "Self-initiated corrective actions implemented (date range)": (
        "Self-initiated intervention",
        "Exclude",
        "Records approximate intervention timing."
    ),
    "Self-initiated corrective actions notes": (
        "Self-initiated intervention",
        "Exclude",
        "Notes about independent interventions."
    ),

    # Behavioural and fish-related observations
    "Individuals air gulping": (
        "Fish behaviour",
        "Review",
        "May indicate low dissolved oxygen; timing relative to measurement needs review."
    ),
    "Individuals tail splashing": (
        "Fish behaviour",
        "Review",
        "Behavioural observation; timing and interpretation need review."
    ),
    "Dead fish": (
        "Fish health outcome",
        "Review",
        "Records mortality since the previous farm visit."
    ),
    "Notes (mortalities)": (
        "Fish health notes",
        "Exclude from primary model",
        "Free-text mortality information."
    ),
    "Feed amount (kg)": (
        "Farm management",
        "Review",
        "Timing and availability at prediction time need verification."
    ),
    "Stocking density (per acre)": (
        "Farm management",
        "Review",
        "Potential pond-level context."
    ),
    "Species": (
        "Farm management",
        "Review",
        "Potential pond-level context."
    ),
    "Weight": (
        "Fish-related information",
        "Review",
        "Self-reported individual fish weight."
    ),
    "Notes": (
        "Free-text notes",
        "Exclude from primary model",
        "Unstructured text may contain outcome or intervention information."
    ),
    "Have fish been helped": (
        "Impact/intervention outcome",
        "Exclude pending timing verification",
        "Near-constant field; exact timing relative to measurement needs verification."
    ),
}

role_rows = []

for column in df.columns:
    if column in column_roles:
        category, eligibility, rationale = column_roles[column]
    else:
        category = "Unclassified"
        eligibility = "Review required"
        rationale = "No classification has been assigned."

    role_rows.append({
        "Column": column,
        "Analytical role": category,
        "Preliminary eligibility": eligibility,
        "Rationale": rationale
    })

column_role_report = pd.DataFrame(role_rows)

print("COLUMN ROLE AUDIT")
display(column_role_report.to_string(index=False))

print("\nCLASSIFICATION SUMMARY")
display(
    column_role_report.groupby(
        ["Analytical role", "Preliminary eligibility"]
    ).size().reset_index(name="Number of columns").to_string(index=False)
)

print("\nUNCLASSIFIED COLUMNS")
unclassified = column_role_report[
    column_role_report["Analytical role"] == "Unclassified"
]

if unclassified.empty:
    print("None. All dataset columns have been assigned a preliminary role.")
else:
    display(unclassified.to_string(index=False))

print("\nCOLUMN COUNT CHECK")
print("Dataset columns:", len(df.columns))
print("Classified columns:", len(column_role_report))
print("Unclassified columns:", len(unclassified))

COLUMN ROLE AUDIT


'                                                       Column             Analytical role             Preliminary eligibility                                                                       Rationale\n                                                      pond_id                  Identifier      Exclude from direct prediction                            Identifies repeated observations from the same pond.\n                                                       region         Geographic metadata                              Review                   May capture regional differences; assess generalisation risk.\n                                      Date of data collection              Visit metadata                              Review               May capture temporal trends and changes in measurement protocols.\n                                      Time of data collection              Visit metadata                              Review                                     Potential


CLASSIFICATION SUMMARY


'            Analytical role             Preliminary eligibility  Number of columns\n    Administrative metadata                              Review                  1\n          Corrective action                             Exclude                 12\n      Environmental context                              Review                  1\n            Farm management                              Review                  3\n             Fish behaviour                              Review                  2\n          Fish health notes          Exclude from primary model                  1\n        Fish health outcome                              Review                  1\n   Fish-related information                              Review                  1\n          Follow-up process                             Exclude                  2\n            Free-text notes          Exclude from primary model                  1\n        Geographic metadata                              Review            


UNCLASSIFIED COLUMNS
None. All dataset columns have been assigned a preliminary role.

COLUMN COUNT CHECK
Dataset columns: 53
Classified columns: 53
Unclassified columns: 0


In [7]:
# AQUA: Audit fish-health and intervention-related fields

audit_columns = [
    "Dead fish",
    "Individuals air gulping",
    "Individuals tail splashing",
    "Have fish been helped",
    "Is follow up",
    "Water quality improved after corrective actions"
]

print("FISH-HEALTH AND INTERVENTION FIELD AUDIT")

for column in audit_columns:
    print(f"\n{'=' * 75}")
    print(f"COLUMN: {column}")
    print(f"{'=' * 75}")

    print("Missing values:", df[column].isna().sum())
    print("Non-missing values:", df[column].notna().sum())
    print("Unique values:", df[column].nunique())

    print("\nMost common recorded values:")
    display(
        df[column]
        .value_counts(dropna=False)
        .head(15)
        .to_frame("Record count")
    )

    print("\nDistribution by target:")
    display(
        pd.crosstab(
            df[column],
            df["Is WQ in range?"],
            margins=True,
            dropna=False
        )
    )

FISH-HEALTH AND INTERVENTION FIELD AUDIT

COLUMN: Dead fish
Missing values: 3908
Non-missing values: 8523
Unique values: 36

Most common recorded values:


,Record count
Dead fish,
0.00,8429
NaN,3908
1.00,11
2.00,7
3.00,7
50.00,6
20.00,6
5.00,5
200.00,5



Distribution by target:


Is WQ in range?,No,Yes,All
Dead fish,,,
0.00,979,7450,8429
1.00,3,8,11
2.00,6,1,7
3.00,6,1,7
4.00,2,1,3
5.00,2,3,5
6.00,1,3,4
7.00,2,2,4
8.00,0,1,1



COLUMN: Individuals air gulping
Missing values: 3004
Non-missing values: 9427
Unique values: 34

Most common recorded values:


,Record count
Individuals air gulping,
0.00,9334
NaN,3004
100.00,11
20.00,8
30.00,7
1000.00,6
10.00,6
50.00,6
500.00,5


... (14 outputs left out)


COLUMN: Water quality improved after corrective actions
Missing values: 11447
Non-missing values: 984
Unique values: 2

Most common recorded values:


,Record count
Water quality improved after corrective actions,
NaN,11447
Yes,828
No,156



Distribution by target:


Is WQ in range?,No,Yes,All
Water quality improved after corrective actions,,,
No,147,9,156
Yes,21,807,828
NaN,2199,9248,11447
All,2367,10064,12431


In [8]:
# AQUA: Investigate records where fish were reported as helped

helped_records = df.loc[
    df["Have fish been helped"].eq("Yes")
].copy()

investigation_columns = [
    "Date of data collection",
    "Time of data collection",
    "pond_id",
    "region",
    "Type",
    "Pond status",
    "Is follow up",
    "Is WQ in range?",
    "Have fish been helped",
    "Corrective actions",
    "Corrective actions requested",
    "Corrective actions implemented",
    "Corrective actions implementation date",
    "Corrective actions taken",
    "Self-initiated corrective actions taken",
    "Self-initiated corrective actions implemented on (exact date)",
    "Self-initiated corrective actions implemented (date range)"
]

print("=" * 85)
print("INVESTIGATION OF RECORDS WHERE FISH WERE REPORTED AS HELPED")
print("=" * 85)

print("Number of records:", len(helped_records))
print("Unique ponds:", helped_records["pond_id"].nunique())

print("\nTARGET DISTRIBUTION")
display(helped_records["Is WQ in range?"].value_counts(dropna=False).to_frame("Count"))

print("\nRECORD DETAILS")
display(
    helped_records[investigation_columns]
    .sort_values(["pond_id", "Date of data collection", "Time of data collection"])
    .to_string(index=False)
)

INVESTIGATION OF RECORDS WHERE FISH WERE REPORTED AS HELPED
Number of records: 57
Unique ponds: 44

TARGET DISTRIBUTION


,Count
Is WQ in range?,
Yes,57



RECORD DETAILS


'Date of data collection Time of data collection       pond_id  region    Type Pond status Is follow up Is WQ in range? Have fish been helped Corrective actions Corrective actions requested Corrective actions implemented Corrective actions implementation date                                            Corrective actions taken Self-initiated corrective actions taken Self-initiated corrective actions implemented on (exact date) Self-initiated corrective actions implemented (date range)\n             06/12/2025                   06:45 ara2_0677080b   Eluru Morning   Treatment          Yes             Yes                   Yes                NaN                          NaN               Yes, all of them                             06/10/2025                                                    Jaggery, Zeolite                                     NaN                                                           NaN                                                        NaN\n             07/07/20

In [9]:
# AQUA: Investigate ammonia measurement protocol across time

ammonia_columns = [
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)"
]

ammonia_audit = df.copy()

ammonia_audit["parsed_date"] = pd.to_datetime(
    ammonia_audit["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

ammonia_audit["month"] = (
    ammonia_audit["parsed_date"].dt.to_period("M").astype(str)
)

print("=" * 85)
print("AMMONIA MEASUREMENT PROTOCOL TRANSITION AUDIT")
print("=" * 85)

print("\n1. OVERALL AVAILABILITY")

display(
    ammonia_audit[ammonia_columns]
    .notna()
    .sum()
    .to_frame("Available records")
)

print("\n2. FIRST AND LAST RECORDED DATE FOR EACH AMMONIA COLUMN")

for column in ammonia_columns:
    available_dates = ammonia_audit.loc[
        ammonia_audit[column].notna(),
        "parsed_date"
    ]

    print(f"\n{column}")
    print("Available records:", len(available_dates))

    if not available_dates.empty:
        print("First recorded date:", available_dates.min().date())
        print("Last recorded date :", available_dates.max().date())

print("\n3. MONTHLY AMMONIA MEASUREMENT AVAILABILITY")

monthly_availability = (
    ammonia_audit
    .groupby("month")[ammonia_columns]
    .count()
)

display(monthly_availability.to_string())

print("\n4. AMMONIA COLUMN OVERLAP")

availability_flags = ammonia_audit[ammonia_columns].notna()

overlap_summary = (
    availability_flags
    .value_counts()
    .rename("Record count")
    .to_frame()
)

display(overlap_summary)

print("\n5. AMMONIA AVAILABILITY BY MEASUREMENT TYPE")

for column in ammonia_columns:
    print(f"\n{column}")

    display(
        pd.crosstab(
            ammonia_audit["Type"],
            ammonia_audit[column].notna(),
            margins=True
        )
    )

print("\n" + "=" * 85)
print("AMMONIA PROTOCOL AUDIT COMPLETED")
print("=" * 85)

AMMONIA MEASUREMENT PROTOCOL TRANSITION AUDIT

1. OVERALL AVAILABILITY


,Available records
Ammonia—TAN (NH3-N) (mg/L),5292
Ammonia—TAN (NH3) (mg/L),1659
Ammonia—NH3 (mg/L),1660



2. FIRST AND LAST RECORDED DATE FOR EACH AMMONIA COLUMN

Ammonia—TAN (NH3-N) (mg/L)
Available records: 5292
First recorded date: 2021-06-19
Last recorded date : 2025-08-19

Ammonia—TAN (NH3) (mg/L)
Available records: 1659
First recorded date: 2025-08-01
Last recorded date : 2026-08-06

Ammonia—NH3 (mg/L)
Available records: 1660
First recorded date: 2025-07-31
Last recorded date : 2026-08-06

3. MONTHLY AMMONIA MEASUREMENT AVAILABILITY


'         Ammonia—TAN (NH3-N) (mg/L)  Ammonia—TAN (NH3) (mg/L)  Ammonia—NH3 (mg/L)\nmonth                                                                            \n2021-06                           1                         0                   0\n2021-07                          30                         0                   0\n2021-08                          54                         0                   0\n2021-09                          50                         0                   0\n2021-10                          59                         0                   0\n2021-11                          51                         0                   0\n2021-12                          66                         0                   0\n2022-01                          70                         0                   0\n2022-02                          90                         0                   0\n2022-03                         108                         0                   0\n202


4. AMMONIA COLUMN OVERLAP


Record count
Ammonia—TAN (NH3-N) (mg/L) Ammonia—TAN (NH3) (mg/L) Ammonia—NH3 (mg/L)              
False                      False                    False                       5481
True                       False                    False                       5290
False                      True                     True                        1658
True                       True                     True                           1
                           False                    True                           1

... (3 outputs left out)

Ammonia—TAN (NH3) (mg/L),False,True,All
Type,,,
Evening,5975,0,5975
Morning,4795,1659,6454
All,10770,1659,12429



Ammonia—NH3 (mg/L)


Ammonia—NH3 (mg/L),False,True,All
Type,,,
Evening,5975,0,5975
Morning,4794,1660,6454
All,10769,1660,12429



AMMONIA PROTOCOL AUDIT COMPLETED


In [10]:
# AQUA: Numeric measurement validity and outlier audit

import numpy as np
import pandas as pd

audit = df.copy()

audit["parsed_date"] = pd.to_datetime(
    audit["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

numeric_columns = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Temp (in °C)",
    "TDS (ppt)"
]

for col in numeric_columns:
    audit[col] = pd.to_numeric(audit[col], errors="coerce")

results = []

for col in numeric_columns:
    values = audit[col].dropna()

    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    results.append({
        "Measurement": col,
        "Available": len(values),
        "Missing": audit[col].isna().sum(),
        "Minimum": values.min(),
        "1st percentile": values.quantile(0.01),
        "Median": values.median(),
        "3rd quartile": q3,
        "99th percentile": values.quantile(0.99),
        "Maximum": values.max(),
        "Zero values": (values == 0).sum(),
        "Negative values": (values < 0).sum(),
        "IQR outliers": ((values < lower_fence) | (values > upper_fence)).sum()
    })

summary = pd.DataFrame(results)

# Documented range checks for core measurements

audit["DO_expected_range_violation"] = False

morning = audit["Type"].eq("Morning")
evening = audit["Type"].eq("Evening")

audit.loc[morning, "DO_expected_range_violation"] = (
    (audit.loc[morning, "DO (mg/L)"] < 3) |
    (audit.loc[morning, "DO (mg/L)"] > 5)
)

audit.loc[evening, "DO_expected_range_violation"] = (
    (audit.loc[evening, "DO (mg/L)"] < 8) |
    (audit.loc[evening, "DO (mg/L)"] > 12)
)

audit["pH_expected_range_violation"] = (
    (audit["pH"] < 6.5) | (audit["pH"] > 8.5)
)

print("=" * 100)
print("AQUA — NUMERIC MEASUREMENT VALIDITY AND OUTLIER AUDIT")
print("=" * 100)

print("\n1. NUMERIC MEASUREMENT SUMMARY")
print(summary.round(3).to_string(index=False))

print("\n2. DOCUMENTED RANGE CHECKS")
print(
    f"DO outside expected Type-specific range: "
    f"{audit['DO_expected_range_violation'].sum()}"
)
print(
    f"pH outside documented range (6.5–8.5): "
    f"{audit['pH_expected_range_violation'].sum()}"
)

print("\n3. DO RANGE VIOLATIONS BY MEASUREMENT TYPE")
print(
    audit.loc[audit["DO_expected_range_violation"]]
    .groupby("Type")
    .size()
    .rename("Records")
    .to_string()
)

print("\n4. pH RANGE VIOLATIONS BY MEASUREMENT TYPE")
print(
    audit.loc[audit["pH_expected_range_violation"]]
    .groupby("Type")
    .size()
    .rename("Records")
    .to_string()
)

print("\n5. EXAMPLE RECORDS OUTSIDE DOCUMENTED RANGES")

violation_examples = audit.loc[
    audit["DO_expected_range_violation"] |
    audit["pH_expected_range_violation"],
    [
        "Date of data collection",
        "pond_id",
        "Type",
        "DO (mg/L)",
        "pH",
        "Is WQ in range?"
    ]
].head(15)

if violation_examples.empty:
    print("No violations found.")
else:
    print(violation_examples.to_string(index=False))

print("\n6. AUDIT NOTES")
print("- IQR outliers are statistical flags, not confirmed measurement errors.")
print("- DO limits are checked separately for Morning and Evening records.")
print("- pH is checked against the documented 6.5–8.5 range.")
print("- Ammonia measurements require protocol-specific interpretation.")
print("- No values are changed, removed, or imputed in this audit.")

print("\n" + "=" * 100)
print("NUMERIC MEASUREMENT AUDIT COMPLETED")
print("=" * 100)

AQUA — NUMERIC MEASUREMENT VALIDITY AND OUTLIER AUDIT

1. NUMERIC MEASUREMENT SUMMARY
               Measurement  Available  Missing  Minimum  1st percentile  Median  3rd quartile  99th percentile  Maximum  Zero values  Negative values  IQR outliers
                 DO (mg/L)      12412       19     0.00            1.00    4.70         10.10            14.60    27.10            1                0            13
                        pH      12412       19     5.52            7.45    8.25          8.40             8.90     9.54            0                0           343
         Turbidity (in cm)       4898     7533     0.00           15.00   29.00         32.00            57.00   132.00           12                0           188
Ammonia—TAN (NH3-N) (mg/L)       5292     7139     0.00            0.00    0.12          0.26             0.89     3.00         1052                0           116
  Ammonia—TAN (NH3) (mg/L)       1659    10772     0.00            0.00    0.25          0.25 

In [11]:
# AQUA: Investigate documented range mismatches and extreme measurements

import pandas as pd
import numpy as np

audit = df.copy()

audit["parsed_date"] = pd.to_datetime(
    audit["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

numeric_columns = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Temp (in °C)",
    "TDS (ppt)"
]

for col in numeric_columns:
    audit[col] = pd.to_numeric(audit[col], errors="coerce")

morning = audit["Type"].eq("Morning")
evening = audit["Type"].eq("Evening")

audit["DO_range_violation"] = False

audit.loc[morning, "DO_range_violation"] = (
    (audit.loc[morning, "DO (mg/L)"] < 3) |
    (audit.loc[morning, "DO (mg/L)"] > 5)
)

audit.loc[evening, "DO_range_violation"] = (
    (audit.loc[evening, "DO (mg/L)"] < 8) |
    (audit.loc[evening, "DO (mg/L)"] > 12)
)

audit["pH_range_violation"] = (
    (audit["pH"] < 6.5) | (audit["pH"] > 8.5)
)

print("=" * 95)
print("AQUA — RANGE MISMATCH AND EXTREME VALUE INVESTIGATION")
print("=" * 95)

print("\n1. DO DOCUMENTED-RANGE VIOLATIONS VS TARGET")

print(pd.crosstab(
    audit["DO_range_violation"],
    audit["Is WQ in range?"],
    margins=True
).to_string())

print("\n2. pH DOCUMENTED-RANGE VIOLATIONS VS TARGET")

print(pd.crosstab(
    audit["pH_range_violation"],
    audit["Is WQ in range?"],
    margins=True
).to_string())

print("\n3. pH VIOLATIONS WITH MISSING MEASUREMENT TYPE")

print(
    audit.loc[
        audit["pH_range_violation"] & audit["Type"].isna(),
        [
            "Date of data collection",
            "Time of data collection",
            "pond_id",
            "DO (mg/L)",
            "pH",
            "Is WQ in range?"
        ]
    ].to_string(index=False)
)

print("\n4. EXTREME TDS OBSERVATIONS — TOP 15")

print(
    audit.nlargest(15, "TDS (ppt)")[
        [
            "Date of data collection",
            "pond_id",
            "Type",
            "TDS (ppt)",
            "DO (mg/L)",
            "pH",
            "Is WQ in range?"
        ]
    ].to_string(index=False)
)

print("\n5. EXTREME DO OBSERVATIONS — TOP 10 HIGHEST AND LOWEST")

do_columns = [
    "Date of data collection",
    "pond_id",
    "Type",
    "DO (mg/L)",
    "pH",
    "Is WQ in range?"
]

print("\nLowest DO:")
print(
    audit.nsmallest(10, "DO (mg/L)")[do_columns]
    .to_string(index=False)
)

print("\nHighest DO:")
print(
    audit.nlargest(10, "DO (mg/L)")[do_columns]
    .to_string(index=False)
)

print("\n6. EXTREME pH OBSERVATIONS — TOP 10 HIGHEST AND LOWEST")

ph_columns = [
    "Date of data collection",
    "pond_id",
    "Type",
    "pH",
    "DO (mg/L)",
    "Is WQ in range?"
]

print("\nLowest pH:")
print(
    audit.nsmallest(10, "pH")[ph_columns]
    .to_string(index=False)
)

print("\nHighest pH:")
print(
    audit.nlargest(10, "pH")[ph_columns]
    .to_string(index=False)
)

print("\n7. EXTREME TEMPERATURE OBSERVATIONS — TOP 10 HIGHEST")

print(
    audit.nlargest(10, "Temp (in °C)")[
        [
            "Date of data collection",
            "pond_id",
            "Type",
            "Temp (in °C)",
            "Is WQ in range?"
        ]
    ].to_string(index=False)
)

print("\n8. AUDIT INTERPRETATION")

print("- A range violation does not automatically mean a measurement is erroneous.")
print("- Target disagreements require investigation; do not relabel observations.")
print("- Extreme TDS values require source-level verification before any correction.")
print("- No observations are deleted, modified, or imputed.")

print("\n" + "=" * 95)
print("RANGE MISMATCH AND EXTREME VALUE INVESTIGATION COMPLETED")
print("=" * 95)

AQUA — RANGE MISMATCH AND EXTREME VALUE INVESTIGATION

1. DO DOCUMENTED-RANGE VIOLATIONS VS TARGET
Is WQ in range?       No    Yes    All
DO_range_violation                    
False                818  10029  10847
True                1549     35   1584
All                 2367  10064  12431

2. pH DOCUMENTED-RANGE VIOLATIONS VS TARGET
Is WQ in range?       No    Yes    All
pH_range_violation                    
False               1570  10064  11634
True                 797      0    797
All                 2367  10064  12431

3. pH VIOLATIONS WITH MISSING MEASUREMENT TYPE
Date of data collection Time of data collection       pond_id  DO (mg/L)   pH Is WQ in range?
             07/14/2021                   00:00 ara2_17b9d7b4       8.50 8.63              No

4. EXTREME TDS OBSERVATIONS — TOP 15
Date of data collection       pond_id    Type  TDS (ppt)  DO (mg/L)   pH Is WQ in range?
             04/01/2022 ara2_20dd0295 Morning     936.00       5.00 8.43              No
             0

In [12]:
# AQUA: Investigate target mismatches and extreme TDS measurements

import pandas as pd
import numpy as np

audit = df.copy()

audit["parsed_date"] = pd.to_datetime(
    audit["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

for col in ["DO (mg/L)", "pH", "TDS (ppt)"]:
    audit[col] = pd.to_numeric(audit[col], errors="coerce")

morning = audit["Type"].eq("Morning")
evening = audit["Type"].eq("Evening")

audit["DO_range_violation"] = False

audit.loc[morning, "DO_range_violation"] = (
    (audit.loc[morning, "DO (mg/L)"] < 3) |
    (audit.loc[morning, "DO (mg/L)"] > 5)
)

audit.loc[evening, "DO_range_violation"] = (
    (audit.loc[evening, "DO (mg/L)"] < 8) |
    (audit.loc[evening, "DO (mg/L)"] > 12)
)

print("=" * 100)
print("AQUA — TARGET MISMATCH AND EXTREME TDS INVESTIGATION")
print("=" * 100)

# 1. DO violations labelled Yes

print("\n1. DO VIOLATIONS LABELLED YES")

do_violation_yes = audit[
    audit["DO_range_violation"] &
    audit["Is WQ in range?"].eq("Yes")
]

print("Record count:", len(do_violation_yes))

print(
    do_violation_yes[
        [
            "Date of data collection",
            "pond_id",
            "Type",
            "DO (mg/L)",
            "pH",
            "TDS (ppt)",
            "Is WQ in range?"
        ]
    ].to_string(index=False)
)

# 2. DO within range but target No

print("\n2. DO WITHIN EXPECTED RANGE BUT LABELLED NO")

do_within_no = audit[
    ~audit["DO_range_violation"] &
    audit["Is WQ in range?"].eq("No")
]

print("Record count:", len(do_within_no))

print(
    do_within_no[
        [
            "Date of data collection",
            "pond_id",
            "Type",
            "DO (mg/L)",
            "pH",
            "TDS (ppt)",
            "Is WQ in range?",
            "Parameters out of range"
        ]
    ].head(30).to_string(index=False)
)

# 3. Extreme TDS records

print("\n3. TDS VALUES ABOVE 50 PPT")

extreme_tds = audit[audit["TDS (ppt)"] > 50].copy()

print("Record count:", len(extreme_tds))
print("Unique ponds:", extreme_tds["pond_id"].nunique())

print(
    extreme_tds[
        [
            "Date of data collection",
            "pond_id",
            "Type",
            "TDS (ppt)",
            "DO (mg/L)",
            "pH",
            "Is WQ in range?"
        ]
    ].sort_values("TDS (ppt)", ascending=False)
    .to_string(index=False)
)

# 4. Repeated extreme TDS values within ponds

print("\n4. PONDS WITH REPEATED TDS VALUES ABOVE 50 PPT")

repeated_extreme_tds = (
    extreme_tds.groupby("pond_id")
    .agg(
        extreme_records=("TDS (ppt)", "size"),
        distinct_tds_values=("TDS (ppt)", "nunique"),
        maximum_tds=("TDS (ppt)", "max"),
        first_date=("parsed_date", "min"),
        last_date=("parsed_date", "max")
    )
    .query("extreme_records > 1")
    .sort_values("extreme_records", ascending=False)
)

if repeated_extreme_tds.empty:
    print("No ponds have repeated extreme TDS records.")
else:
    print(repeated_extreme_tds.to_string())

# 5. Nearby observations for the ponds with highest TDS

print("\n5. SAME-POND OBSERVATIONS AROUND THE HIGHEST TDS RECORDS")

top_tds = audit.nlargest(5, "TDS (ppt)")

for _, row in top_tds.iterrows():
    pond = row["pond_id"]
    date = row["parsed_date"]

    nearby = audit[
        (audit["pond_id"] == pond) &
        (audit["parsed_date"].between(
            date - pd.Timedelta(days=45),
            date + pd.Timedelta(days=45)
        ))
    ].sort_values("parsed_date")

    print(
        f"\nPond: {pond} | Extreme date: {date.date()} | "
        f"TDS: {row['TDS (ppt)']}"
    )

    print(
        nearby[
            [
                "Date of data collection",
                "Type",
                "TDS (ppt)",
                "DO (mg/L)",
                "pH",
                "Is WQ in range?"
            ]
        ].to_string(index=False)
    )

print("\n" + "=" * 100)
print("TARGET MISMATCH AND EXTREME TDS INVESTIGATION COMPLETED")
print("=" * 100)

AQUA — TARGET MISMATCH AND EXTREME TDS INVESTIGATION

1. DO VIOLATIONS LABELLED YES
Record count: 35
Date of data collection       pond_id    Type  DO (mg/L)   pH  TDS (ppt) Is WQ in range?
             08/06/2026 ara2_e74f26b3 Evening       6.90 8.44        NaN             Yes
             08/06/2026 ara2_46286ebb Evening       6.70 8.23        NaN             Yes
             08/03/2026 ara2_b0a5ed5f Evening      14.00 8.47        NaN             Yes
             08/03/2026 ara2_fea90a74 Evening      14.40 8.38        NaN             Yes
             08/03/2026 ara2_b0a5ed5f Morning       5.30 8.32        NaN             Yes
             07/31/2026 ara2_8ee1e5e2 Evening      14.10 8.43        NaN             Yes
             07/31/2026 ara2_296bcb81 Evening      12.50 8.38        NaN             Yes
             07/31/2026 ara2_ea0cc887 Evening      13.20 8.47        NaN             Yes
             07/21/2026 ara2_d60f166f Evening      12.30 8.42        NaN             Yes
         

In [13]:
# AQUA: Investigate source context of extreme TDS measurements

tds_context_columns = [
    "Date of data collection",
    "Time of data collection",
    "pond_id",
    "region",
    "Type",
    "TDS (ppt)",
    "DO (mg/L)",
    "pH",
    "Observer",
    "Equipment",
    "Notes",
    "Is WQ in range?"
]

extreme_tds_records = df.loc[
    df["TDS (ppt)"] > 50,
    tds_context_columns
].sort_values(
    by="TDS (ppt)",
    ascending=False
)

print("=" * 100)
print("EXTREME TDS SOURCE-CONTEXT INVESTIGATION")
print("=" * 100)

print("\n1. EXTREME TDS RECORDS WITH OBSERVER, EQUIPMENT AND NOTES")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", 100,
    "display.width", 200
):
    display(extreme_tds_records)

print("\n2. EXTREME TDS COUNTS BY OBSERVER")

display(
    extreme_tds_records["Observer"]
    .value_counts(dropna=False)
    .to_frame("Record count")
)

print("\n3. EXTREME TDS COUNTS BY EQUIPMENT")

display(
    extreme_tds_records["Equipment"]
    .value_counts(dropna=False)
    .to_frame("Record count")
)

print("\n" + "=" * 100)
print("TDS SOURCE-CONTEXT INVESTIGATION COMPLETED")
print("=" * 100)

EXTREME TDS SOURCE-CONTEXT INVESTIGATION

1. EXTREME TDS RECORDS WITH OBSERVER, EQUIPMENT AND NOTES


,Date of data collection,Time of data collection,pond_id,region,Type,TDS (ppt),DO (mg/L),pH,Observer,Equipment,Notes,Is WQ in range?
11149,04/01/2022,07:57,ara2_20dd0295,Eluru,Morning,936.00,5.00,8.43,Nagaraju P.,Hanna,NaN,No
12216,09/20/2021,06:10,ara2_bd32b7fb,Eluru,Morning,923.00,2.70,7.81,unknown,NaN,NaN,No
9791,10/27/2022,06:55,ara2_a2d65cb4,Eluru,Morning,795.00,2.60,8.29,Nagaraju P.,Taiwan #2,NaN,No
12215,09/20/2021,18:25,ara2_bd32b7fb,Eluru,Evening,795.00,5.70,8.12,unknown,NaN,NaN,No
9838,10/17/2022,17:17,ara2_2d4cda6c,Eluru,Evening,781.00,12.00,8.28,Maneesh,Taiwan,NaN,Yes
9847,10/17/2022,07:10,ara2_01463d19,Eluru,Morning,781.00,2.80,7.74,Maneesh,Taiwan,NaN,No
9836,10/17/2022,17:43,ara2_87661414,Eluru,Evening,555.00,5.20,8.48,Nagaraju P.,Hanna,NaN,No
9787,10/27/2022,07:58,ara2_87661414,Eluru,Morning,543.00,2.70,8.27,Nagaraju P.,Taiwan #2,NaN,No
7537,12/18/2023,16:10,ara2_03d81c1d,Eluru,Evening,155.00,9.60,8.28,Sanjay,Taiwan #2,NaN,Yes
7564,12/15/2023,06:36,ara2_a0d5d74d,Eluru,Morning,143.00,3.10,8.26,Sanjay,Taiwan #9,NaN,Yes



2. EXTREME TDS COUNTS BY OBSERVER


,Record count
Observer,
NaN,12
Nagaraju P.,5
unknown,3
Maneesh,2
Sanjay,2



3. EXTREME TDS COUNTS BY EQUIPMENT


,Record count
Equipment,
Taiwan #8,9
NaN,3
Taiwan #2,3
Taiwan #7,3
Hanna,2
Taiwan,2
Taiwan #9,1
"Taiwan #2, Winkler's Method (DO)",1



TDS SOURCE-CONTEXT INVESTIGATION COMPLETED


In [14]:
# AQUA: Verify TDS units in official documentation

from pathlib import Path

documentation_dir = (
    Path.cwd().parent / "docs" / "source_reference"
)

documentation_files = [
    documentation_dir / "data_dictionary.md",
    documentation_dir / "official_README.md"
]

search_terms = [
    "TDS",
    "total dissolved solids",
    "ppt",
    "ppm",
    "conductivity"
]

print("=" * 100)
print("AQUA — OFFICIAL TDS DOCUMENTATION VERIFICATION")
print("=" * 100)

for file_path in documentation_files:
    print(f"\nDOCUMENT: {file_path.name}")
    print("-" * 100)

    if not file_path.exists():
        print("FILE NOT FOUND:", file_path)
        continue

    lines = file_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    matched_lines = set()

    for index, line in enumerate(lines):
        if any(term.lower() in line.lower() for term in search_terms):
            for nearby_index in range(
                max(0, index - 2),
                min(len(lines), index + 3)
            ):
                matched_lines.add(nearby_index)

    if matched_lines:
        for index in sorted(matched_lines):
            print(f"Line {index + 1}: {lines[index]}")
    else:
        print("No matching references found.")

print("\n" + "=" * 100)
print("TDS DOCUMENTATION VERIFICATION COMPLETED")
print("=" * 100)

AQUA — OFFICIAL TDS DOCUMENTATION VERIFICATION

DOCUMENT: data_dictionary.md
----------------------------------------------------------------------------------------------------
Line 51: | `Ammonia—NH3 (mg/L)` | Unionized ammonia (NH₃) in mg/L. New primary metric from August 2025—the directly toxic form. |
Line 52: | `Temp (in °C)` | Water temperature in Celsius |
Line 53: | `TDS (ppt)` | Total dissolved solids in parts per thousand |
Line 54: | `Alkalinity (mg/L)` | Alkalinity in mg/L |
Line 55: | `Hardness (mg/L)` | Water hardness in mg/L |

DOCUMENT: official_README.md
----------------------------------------------------------------------------------------------------


No matching references found.

TDS DOCUMENTATION VERIFICATION COMPLETED


In [15]:
# AQUA: Categorical Value Consistency Audit

print("=" * 100)
print("AQUA — CATEGORICAL VALUE CONSISTENCY AUDIT")
print("=" * 100)

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(f"\nCategorical columns identified: {len(categorical_columns)}")

for col in categorical_columns:

    series = df[col].dropna().astype(str)

    unique_count = series.nunique()

    whitespace_count = (
        series != series.str.strip()
    ).sum()

    print("\n" + "-" * 100)
    print(f"COLUMN: {col}")
    print(f"Non-missing records: {len(series):,}")
    print(f"Unique values: {unique_count:,}")
    print(f"Leading/trailing whitespace records: {whitespace_count:,}")

    if unique_count <= 30:
        print("\nValue frequencies:")
        print(series.value_counts().to_string())

    else:
        print("\nTop 10 values:")
        print(series.value_counts().head(10).to_string())

print("\n" + "=" * 100)
print("CATEGORICAL VALUE CONSISTENCY AUDIT COMPLETED")
print("=" * 100)

AQUA — CATEGORICAL VALUE CONSISTENCY AUDIT

Categorical columns identified: 37

----------------------------------------------------------------------------------------------------
COLUMN: pond_id
Non-missing records: 12,431
Unique values: 317
Leading/trailing whitespace records: 0

Top 10 values:
pond_id
ara2_09faa72c    170
ara2_ff85ba57    140
ara2_3587929e    123
ara2_cd998399    120
ara2_725e7f3b    117
ara2_31afc592    106
ara2_c4feda19    105
ara2_8ae86d8e    104
ara2_f5c44392    103
ara2_9ea43968    103



----------------------------------------------------------------------------------------------------
COLUMN: region
Non-missing records: 12,431
Unique values: 2
Leading/trailing whitespace records: 0

Value frequencies:
region
Eluru      9248
Nellore    3183

----------------------------------------------------------------------------------------------------
COLUMN: Date of data collection
Non-missing records: 12,431
Unique values: 1,210
Leading/trailing whitespace records: 0

Top 10 values:
Date of data collection
05/20/2026    30
09/30/2025    30
03/12/2026    28
09/15/2025    28
09/12/2025    28
04/03/2025    28
04/02/2025    28
11/26/2023    28
01/02/2026    27
09/26/2025    27

----------------------------------------------------------------------------------------------------
COLUMN: Time of data collection
Non-missing records: 12,431
Unique values: 356
Leading/trailing whitespace records: 0

Top 10 values:
Time of data collection
07:10    216
07:15    196
16:10    175
07:30    

C:\Users\hymar\AppData\Local\Temp\ipykernel_36140\1473899103.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


In [16]:
# AQUA: Investigate whitespace and duplicate-looking categorical labels

print("=" * 100)
print("AQUA — WHITESPACE AND LABEL COLLISION INVESTIGATION")
print("=" * 100)

for col in df.columns:

    if not (
        pd.api.types.is_object_dtype(df[col])
        or pd.api.types.is_string_dtype(df[col])
        or isinstance(df[col].dtype, pd.CategoricalDtype)
    ):
        continue

    series = df[col].dropna().astype(str)

    whitespace_mask = series.ne(series.str.strip())

    if not whitespace_mask.any():
        continue

    print("\n" + "-" * 100)
    print(f"COLUMN: {col}")
    print(f"Records with surrounding whitespace: {whitespace_mask.sum()}")

    print("\nOriginal values (exact representation):")

    print(
        series[whitespace_mask]
        .value_counts()
        .head(30)
        .rename_axis("Original value")
        .to_string()
    )

    normalised = series.str.strip()

    collision_groups = (
        pd.DataFrame({
            "original": series,
            "trimmed": normalised
        })
        .groupby("trimmed")["original"]
        .agg(lambda values: sorted(set(values)))
    )

    collisions = collision_groups[
        collision_groups.apply(lambda values: len(values) > 1)
    ]

    print("\nValues that become identical after trimming:")

    if collisions.empty:
        print("No duplicate-label collisions detected.")
    else:
        for trimmed_value, original_values in collisions.items():
            print(f"\nStandardised appearance: {trimmed_value!r}")
            print(f"Original forms: {original_values}")

print("\n" + "=" * 100)
print("WHITESPACE INVESTIGATION COMPLETED")
print("=" * 100)

AQUA — WHITESPACE AND LABEL COLLISION INVESTIGATION

----------------------------------------------------------------------------------------------------
COLUMN: Reason follow up not possible
Records with surrounding whitespace: 1

Original values (exact representation):
Original value
Actually today the live Tilapia harvest was going on in the pond     1

Values that become identical after trimming:
No duplicate-label collisions detected.

----------------------------------------------------------------------------------------------------
COLUMN: Corrective actions requested (other)
Records with surrounding whitespace: 1

Original values (exact representation):
Original value
Stop feeding 24 hours     1

Values that become identical after trimming:
No duplicate-label collisions detected.

----------------------------------------------------------------------------------------------------
COLUMN: Corrective actions amount requested
Records with surrounding whitespace: 11

Original valu

In [17]:
# RE-EDITED VERSION OF CELL 17
# Original cell: 17 — categorical_value_consistency_audit
# Purpose: Quantify the effect of trimming whitespace on categorical labels.
# Important: Original dataframe is NOT modified.

print("=" * 100)
print("AQUA — REVISED CATEGORICAL CONSISTENCY SUMMARY")
print("=" * 100)

string_columns = [
    col for col in df.columns
    if (
        pd.api.types.is_object_dtype(df[col])
        or pd.api.types.is_string_dtype(df[col])
    )
]

total_affected_records = 0
total_unique_labels_merged = 0

for col in string_columns:

    original = df[col].dropna().astype(str)
    trimmed = original.str.strip()

    affected = original.ne(trimmed).sum()

    if affected == 0:
        continue

    original_unique = original.nunique()
    trimmed_unique = trimmed.nunique()

    labels_merged = original_unique - trimmed_unique

    total_affected_records += affected
    total_unique_labels_merged += labels_merged

    print("\n" + "-" * 100)
    print(f"Column: {col}")
    print(f"Whitespace-affected records: {affected:,}")
    print(f"Unique labels before trimming: {original_unique:,}")
    print(f"Unique labels after trimming: {trimmed_unique:,}")
    print(f"Unique labels merged: {labels_merged:,}")

print("\n" + "=" * 100)
print("OVERALL SUMMARY")
print("=" * 100)
print(f"Total affected records across columns: {total_affected_records:,}")
print(f"Total unique labels merged across columns: {total_unique_labels_merged:,}")
print("Original dataframe modified: NO")
print("=" * 100)

AQUA — REVISED CATEGORICAL CONSISTENCY SUMMARY

----------------------------------------------------------------------------------------------------
Column: Reason follow up not possible
Whitespace-affected records: 1
Unique labels before trimming: 7
Unique labels after trimming: 7
Unique labels merged: 0

----------------------------------------------------------------------------------------------------
Column: Corrective actions requested (other)
Whitespace-affected records: 1
Unique labels before trimming: 43
Unique labels after trimming: 43
Unique labels merged: 0

----------------------------------------------------------------------------------------------------
Column: Corrective actions amount requested
Whitespace-affected records: 11
Unique labels before trimming: 74
Unique labels after trimming: 71
Unique labels merged: 3

----------------------------------------------------------------------------------------------------
Column: Corrective actions taken (other)
Whitespace-a

In [18]:
# AQUA: Date and Time Consistency Audit

print("=" * 100)
print("AQUA — DATE AND TIME CONSISTENCY AUDIT")
print("=" * 100)

date_col = df["Date of data collection"].astype("string").str.strip()
time_col = df["Time of data collection"].astype("string").str.strip()

parsed_dates = pd.to_datetime(
    date_col,
    format="%m/%d/%Y",
    errors="coerce"
)

parsed_times = pd.to_datetime(
    time_col,
    format="%H:%M",
    errors="coerce"
)

print("\nDATE AUDIT")
print("-" * 100)
print(f"Total records: {len(df):,}")
print(f"Successfully parsed dates: {parsed_dates.notna().sum():,}")
print(f"Unparseable dates: {parsed_dates.isna().sum():,}")
print(f"Earliest date: {parsed_dates.min()}")
print(f"Latest date: {parsed_dates.max()}")
print(f"Unique collection dates: {parsed_dates.nunique():,}")

print("\nTIME AUDIT")
print("-" * 100)
print(f"Successfully parsed times: {parsed_times.notna().sum():,}")
print(f"Unparseable times: {parsed_times.isna().sum():,}")
print(f"Unique collection times: {parsed_times.nunique():,}")

print("\nMOST COMMON COLLECTION TIMES")
print("-" * 100)
print(time_col.value_counts().head(20).to_string())

print("\nEARLIEST AND LATEST COLLECTION TIMES")
print("-" * 100)
valid_times = parsed_times.dropna().dt.strftime("%H:%M")
print(f"Earliest time: {valid_times.min()}")
print(f"Latest time: {valid_times.max()}")

print("\nTIME VALUES BY MEASUREMENT TYPE")
print("-" * 100)

time_type_summary = pd.crosstab(
    df["Type"],
    time_col,
    margins=True
)

for measurement_type in ["Morning", "Evening"]:
    if measurement_type in time_type_summary.index:
        counts = (
            df.loc[df["Type"] == measurement_type, "Time of data collection"]
            .value_counts()
            .sort_index()
        )

        print(f"\n{measurement_type}:")
        print(f"Number of records: {counts.sum():,}")
        print(f"Unique time values: {counts.size:,}")
        print(f"Most common times:\n{counts.sort_values(ascending=False).head(10).to_string()}")

print("\n" + "=" * 100)
print("DATE AND TIME CONSISTENCY AUDIT COMPLETED")
print("=" * 100)

AQUA — DATE AND TIME CONSISTENCY AUDIT

DATE AUDIT
----------------------------------------------------------------------------------------------------
Total records: 12,431
Successfully parsed dates: 12,431
Unparseable dates: 0
Earliest date: 2021-06-19 00:00:00
Latest date: 2026-08-06 00:00:00
Unique collection dates: 1,210

TIME AUDIT
----------------------------------------------------------------------------------------------------
Successfully parsed times: 12,431
Unparseable times: 0
Unique collection times: 356

MOST COMMON COLLECTION TIMES
----------------------------------------------------------------------------------------------------
Time of data collection
07:10    216
07:15    196
16:10    175
07:30    159
06:45    159
07:20    147
16:30    145
07:05    137
16:15    128
16:20    126
06:55    125
16:35    118
07:18    117
06:40    116
06:50    115
06:30    115
16:40    112
07:25    110
16:45    108
17:05     98

EARLIEST AND LATEST COLLECTION TIMES
----------------------

In [19]:
# AQUA: Investigate unusual collection times
# Related audit: Cell 20 — date_and_time_consistency_audit
# Purpose: Identify unusual clock times and investigate their measurement Type.

print("=" * 100)
print("AQUA — UNUSUAL COLLECTION TIME INVESTIGATION")
print("=" * 100)

time_audit = df[
    [
        "Date of data collection",
        "Time of data collection",
        "pond_id",
        "region",
        "Type",
        "Is WQ in range?"
    ]
].copy()

time_audit["_parsed_time"] = pd.to_datetime(
    time_audit["Time of data collection"].astype("string").str.strip(),
    format="%H:%M",
    errors="coerce"
)

time_audit["_hour"] = time_audit["_parsed_time"].dt.hour

print("\nRECORD COUNTS BY CLOCK HOUR AND MEASUREMENT TYPE")
print("-" * 100)

hour_summary = pd.crosstab(
    time_audit["_hour"],
    time_audit["Type"],
    dropna=False
)

print(hour_summary.to_string())

print("\nRECORDS WITH MIDNIGHT TIME (00:00)")
print("-" * 100)

midnight_records = time_audit[
    time_audit["Time of data collection"].astype("string").str.strip() == "00:00"
]

print(f"Total midnight records: {len(midnight_records):,}")

if not midnight_records.empty:
    print(
        midnight_records.drop(columns=["_parsed_time", "_hour"])
        .to_string(index=False)
    )

print("\nMEASUREMENT TYPE MISSINGNESS")
print("-" * 100)

missing_type = time_audit[time_audit["Type"].isna()]

print(f"Records with missing Type: {len(missing_type):,}")

if not missing_type.empty:
    print(
        missing_type.drop(columns=["_parsed_time", "_hour"])
        .to_string(index=False)
    )

print("\nEARLIEST AND LATEST TIMES WITHIN EACH TYPE")
print("-" * 100)

for measurement_type in ["Morning", "Evening"]:
    subset = time_audit[
        time_audit["Type"] == measurement_type
    ]

    if not subset.empty:
        print(f"\n{measurement_type}")
        print(f"Earliest: {subset['_parsed_time'].min().strftime('%H:%M')}")
        print(f"Latest: {subset['_parsed_time'].max().strftime('%H:%M')}")

print("\n" + "=" * 100)
print("UNUSUAL COLLECTION TIME INVESTIGATION COMPLETED")
print("=" * 100)

AQUA — UNUSUAL COLLECTION TIME INVESTIGATION

RECORD COUNTS BY CLOCK HOUR AND MEASUREMENT TYPE
----------------------------------------------------------------------------------------------------
Type   Evening  Morning  NaN
_hour                       
0            4       11    2
3            0        1    0
4            1        0    0
5            0        3    0
6            0     2291    0
7            0     3860    0
8            0      281    0
9            0        6    0
12           1        0    0
13           2        0    0
15         865        0    0
16        3753        0    0
17        1283        0    0
18          65        1    0
20           1        0    0

RECORDS WITH MIDNIGHT TIME (00:00)
----------------------------------------------------------------------------------------------------
Total midnight records: 17
Date of data collection Time of data collection       pond_id  region    Type Is WQ in range?
             10/25/2021                   00:00 ara2_

In [20]:
# AQUA — Cell 22
# Name: unusual_collection_time_investigation
# Purpose: Investigate unusual collection times and missing measurement types.
# Related cell: Cell 20 — date_and_time_consistency_audit

time_audit = df[
    ["Time of data collection", "pond_id", "region", "Type", "Is WQ in range?"]
].copy()

time_audit["_time"] = pd.to_datetime(
    time_audit["Time of data collection"].astype("string").str.strip(),
    format="%H:%M",
    errors="coerce"
)

time_audit["_hour"] = time_audit["_time"].dt.hour

print("AQUA — UNUSUAL COLLECTION TIME INVESTIGATION")

# 1. Records by hour and measurement type
print("\nRECORD COUNTS BY HOUR")

hour_summary = pd.crosstab(
    time_audit["_hour"],
    time_audit["Type"],
    dropna=False
)

print(hour_summary.to_string())

# 2. Midnight records
midnight = time_audit[time_audit["_time"].dt.strftime("%H:%M") == "00:00"]

print("\nMIDNIGHT RECORDS")
print(f"Total: {len(midnight)}")

if not midnight.empty:
    print(
        midnight[
            ["Time of data collection", "pond_id", "region", "Type",
             "Is WQ in range?"]
        ].to_string(index=False)
    )

# 3. Missing measurement type
missing_type = time_audit[time_audit["Type"].isna()]

print("\nRECORDS WITH MISSING TYPE")
print(f"Total: {len(missing_type)}")

if not missing_type.empty:
    print(
        missing_type[
            ["Time of data collection", "pond_id", "region",
             "Is WQ in range?"]
        ].to_string(index=False)
    )

# 4. Time boundaries by measurement type
print("\nTIME BOUNDARIES BY TYPE")

for measurement_type in ["Morning", "Evening"]:
    subset = time_audit[time_audit["Type"] == measurement_type]

    if not subset.empty:
        print(
            f"{measurement_type}: "
            f"{subset['_time'].min().strftime('%H:%M')}–"
            f"{subset['_time'].max().strftime('%H:%M')} "
            f"({len(subset):,} records)"
        )

print("\nCell 21 investigation completed.")

AQUA — UNUSUAL COLLECTION TIME INVESTIGATION

RECORD COUNTS BY HOUR
Type   Evening  Morning  NaN
_hour                       
0            4       11    2
3            0        1    0
4            1        0    0
5            0        3    0
6            0     2291    0
7            0     3860    0
8            0      281    0
9            0        6    0
12           1        0    0
13           2        0    0
15         865        0    0
16        3753        0    0
17        1283        0    0
18          65        1    0
20           1        0    0

MIDNIGHT RECORDS
Total: 17
Time of data collection       pond_id  region    Type Is WQ in range?
                  00:00 ara2_5a3d15dd Nellore Evening              No
                  00:00 ara2_407f70f7 Nellore Evening              No
                  00:00 ara2_c5671cf4 Nellore Morning             Yes
                  00:00 ara2_6c015433 Nellore Morning             Yes
                  00:00 ara2_65a854d9 Nellore Morning        

In [21]:
# AQUA — Cell 23
# Name: temporal_sampling_coverage_audit
# Purpose: Examine data collection coverage across years, months and regions.
# Related cells: Cell 20 — date_and_time_consistency_audit
#                Cell 21–22 — unusual_collection_time_investigation

date_audit = df[
    ["Date of data collection", "region", "pond_id", "Type"]
].copy()

date_audit["_date"] = pd.to_datetime(
    date_audit["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

date_audit["_year"] = date_audit["_date"].dt.year
date_audit["_month"] = date_audit["_date"].dt.month

print("AQUA — TEMPORAL SAMPLING COVERAGE AUDIT")

# 1. Annual records by region
print("\nANNUAL RECORD COUNTS BY REGION")

annual_summary = pd.crosstab(
    date_audit["_year"],
    date_audit["region"],
    margins=True
)

print(annual_summary.to_string())

# 2. Monthly records by year
print("\nMONTHLY RECORD COUNTS")

monthly_summary = pd.crosstab(
    date_audit["_year"],
    date_audit["_month"]
)

print(monthly_summary.to_string())

# 3. Unique ponds and collection dates by year
print("\nANNUAL SAMPLING COVERAGE")

coverage_summary = (
    date_audit.groupby("_year")
    .agg(
        total_records=("pond_id", "size"),
        unique_ponds=("pond_id", "nunique"),
        unique_collection_dates=("_date", "nunique")
    )
)

print(coverage_summary.to_string())

# 4. Annual records by measurement type
print("\nANNUAL RECORD COUNTS BY MEASUREMENT TYPE")

type_summary = pd.crosstab(
    date_audit["_year"],
    date_audit["Type"],
    margins=True
)

print(type_summary.to_string())

print("\nCell 23 completed.")

AQUA — TEMPORAL SAMPLING COVERAGE AUDIT

ANNUAL RECORD COUNTS BY REGION
region  Eluru  Nellore    All
_year                        
2021      332      338    670
2022      893     1319   2212
2023     1229      862   2091
2024     1829      664   2493
2025     3004        0   3004
2026     1961        0   1961
All      9248     3183  12431

MONTHLY RECORD COUNTS
_month   1    2    3    4    5    6    7    8    9    10   11   12
_year                                                             
2021      0    0    0    0    0    2   43   99  114   97  142  173
2022    194  192  219  221  222  208  176  224  156  182  103  115
2023    158  198  181  158  124  132  108  188  186  226  204  228
2024    218  208  190  232  234  127  114  168  254  258  221  269
2025    224  255  259  314  211  221  230  178  316  186  325  285
2026    151  291  404  382  359  117  233   24    0    0    0    0

ANNUAL SAMPLING COVERAGE
       total_records  unique_ponds  unique_collection_dates
_year        

In [22]:
# AQUA — Cell 24
# Name: pond_level_observation_frequency_audit
# Purpose: Examine the distribution of observations across individual ponds.
# Related cell: Cell 23 — temporal_sampling_coverage_audit

print("AQUA — POND-LEVEL OBSERVATION FREQUENCY AUDIT")

# 1. Observation count per pond
pond_frequency = (
    df.groupby(["region", "pond_id"])
    .agg(
        observations=("pond_id", "size"),
        collection_dates=("Date of data collection", "nunique")
    )
    .reset_index()
)

# 2. Overall distribution
print("\nOVERALL POND OBSERVATION DISTRIBUTION")

print(
    pond_frequency["observations"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .round(2)
    .to_string()
)

print(f"\nTotal ponds: {len(pond_frequency):,}")
print(f"Ponds with only 1 observation: {(pond_frequency['observations'] == 1).sum():,}")
print(f"Ponds with more than 10 observations: {(pond_frequency['observations'] > 10).sum():,}")

# 3. Regional comparison
print("\nPOND OBSERVATION FREQUENCY BY REGION")

regional_summary = (
    pond_frequency.groupby("region")
    .agg(
        ponds=("pond_id", "count"),
        total_observations=("observations", "sum"),
        mean_observations=("observations", "mean"),
        median_observations=("observations", "median"),
        minimum=("observations", "min"),
        maximum=("observations", "max")
    )
    .round(2)
)

print(regional_summary.to_string())

# 4. Most frequently observed ponds
print("\nTOP 10 PONDS BY OBSERVATION COUNT")

top_ponds = pond_frequency.nlargest(10, "observations")

print(top_ponds.to_string(index=False))

# 5. Distribution of pond observation counts
print("\nPOND FREQUENCY CATEGORIES")

pond_frequency["frequency_category"] = pd.cut(
    pond_frequency["observations"],
    bins=[0, 5, 10, 25, 50, float("inf")],
    labels=["1–5", "6–10", "11–25", "26–50", "Above 50"]
)

frequency_summary = pd.crosstab(
    pond_frequency["frequency_category"],
    pond_frequency["region"]
)

print(frequency_summary.to_string())

print("\nCell 24 completed.")

AQUA — POND-LEVEL OBSERVATION FREQUENCY AUDIT

OVERALL POND OBSERVATION DISTRIBUTION
count   317.00
mean     39.21
std      30.79
min       1.00
25%      14.00
50%      31.00
75%      60.00
90%      84.00
95%      96.20
max     170.00

Total ponds: 317
Ponds with only 1 observation: 1
Ponds with more than 10 observations: 259

POND OBSERVATION FREQUENCY BY REGION
         ponds  total_observations  mean_observations  median_observations  minimum  maximum
region                                                                                      
Eluru      254                9248              36.41                30.00        1      170
Nellore     63                3183              50.52                48.00        6      123

TOP 10 PONDS BY OBSERVATION COUNT
 region       pond_id  observations  collection_dates
  Eluru ara2_09faa72c           170                89
  Eluru ara2_ff85ba57           140                74
Nellore ara2_3587929e           123                62
  Eluru ara

In [23]:
# AQUA — Cell 25
# Name: pond_level_monitoring_interval_audit
# Purpose: Measure time gaps between consecutive unique collection dates per pond.
# Related audits: Cell 23 (temporal coverage), Cell 24 (pond observation frequency).

import pandas as pd
from pathlib import Path

# 1. Prepare the required columns without modifying the original dataframe.
audit_df = df[
    ["pond_id", "region", "Date of data collection", "Is follow up"]
].copy()

audit_df["collection_date"] = pd.to_datetime(
    audit_df["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

# 2. Reduce to one record per pond per collection date.
# Morning and evening measurements on the same date must not create a zero-day gap.
pond_dates = (
    audit_df.dropna(subset=["pond_id", "collection_date"])
    .groupby(["pond_id", "region", "collection_date"], as_index=False)
    .agg(
        records_on_date=("collection_date", "size"),
        follow_up_records=("Is follow up", lambda x: (x == "Yes").sum())
    )
    .sort_values(["pond_id", "collection_date"])
)

# 3. Calculate the interval between consecutive collection dates for each pond.
pond_dates["gap_days"] = (
    pond_dates.groupby("pond_id")["collection_date"]
    .diff()
    .dt.days
)

# 4. Summarise intervals for each pond.
pond_gap_summary = (
    pond_dates.dropna(subset=["gap_days"])
    .groupby(["pond_id", "region"])
    .agg(
        collection_dates=("collection_date", "count"),
        mean_gap_days=("gap_days", "mean"),
        median_gap_days=("gap_days", "median"),
        minimum_gap_days=("gap_days", "min"),
        maximum_gap_days=("gap_days", "max"),
        gaps_1_to_3_days=("gap_days", lambda x: x.between(1, 3).sum()),
        gaps_over_30_days=("gap_days", lambda x: (x > 30).sum())
    )
    .reset_index()
)

# 5. Overall interval distribution.
all_gaps = pond_dates["gap_days"].dropna()

gap_distribution = pd.Series({
    "Total consecutive-date intervals": len(all_gaps),
    "1–3 days": all_gaps.between(1, 3).sum(),
    "4–7 days": all_gaps.between(4, 7).sum(),
    "8–30 days": all_gaps.between(8, 30).sum(),
    "31–60 days": all_gaps.between(31, 60).sum(),
    "61–90 days": all_gaps.between(61, 90).sum(),
    "Over 90 days": (all_gaps > 90).sum()
})

# 6. Regional interval summary.
regional_gaps = (
    pond_dates.dropna(subset=["gap_days"])
    .groupby("region")["gap_days"]
    .agg(
        interval_count="count",
        mean_gap_days="mean",
        median_gap_days="median",
        minimum_gap_days="min",
        maximum_gap_days="max"
    )
    .round(2)
)

# 7. Save the detailed pond-level results for later review.
output_dir = Path("../outputs/tables")
output_dir.mkdir(parents=True, exist_ok=True)

pond_gap_summary.to_csv(
    output_dir / "pond_level_monitoring_intervals.csv",
    index=False
)

# 8. Print one consolidated output block.
print("AQUA — POND-LEVEL MONITORING INTERVAL AUDIT")
print(f"Unique pond-date combinations: {len(pond_dates):,}")
print(f"Ponds with at least one consecutive-date interval: {len(pond_gap_summary):,}")
print(f"Intervals calculated: {len(all_gaps):,}")

print("\nOVERALL GAP DISTRIBUTION")
print(gap_distribution.to_string())

print("\nOVERALL INTERVAL STATISTICS (DAYS)")
print(
    all_gaps.describe()
    .round(2)
    .to_string()
)

print("\nREGIONAL INTERVAL SUMMARY (DAYS)")
print(regional_gaps.to_string())

print("\nPONDS WITH THE LONGEST MEDIAN MONITORING GAPS")
print(
    pond_gap_summary
    .sort_values("median_gap_days", ascending=False)
    .head(10)
    .round(2)
    .to_string(index=False)
)

print("\nDETAILED RESULTS SAVED:")
print(output_dir / "pond_level_monitoring_intervals.csv")

AQUA — POND-LEVEL MONITORING INTERVAL AUDIT
Unique pond-date combinations: 6,479
Ponds with at least one consecutive-date interval: 304
Intervals calculated: 6,162

OVERALL GAP DISTRIBUTION
Total consecutive-date intervals    6162
1–3 days                             553
4–7 days                             353
8–30 days                           3646
31–60 days                          1296
61–90 days                           152
Over 90 days                         162

OVERALL INTERVAL STATISTICS (DAYS)
count   6162.00
mean      26.08
std       31.23
min        1.00
25%       14.00
50%       21.00
75%       31.00
max     1098.00

REGIONAL INTERVAL SUMMARY (DAYS)
         interval_count  mean_gap_days  median_gap_days  minimum_gap_days  maximum_gap_days
region                                                                                     
Eluru              4604          26.14            22.00              1.00           1098.00
Nellore            1558          25.90           

In [24]:
# AQUA — Cell 26
# RE-EDITED VERSION OF CELL 25
# Name: corrected_pond_level_monitoring_interval_summary
# Purpose: Correct the collection-date count label and verify pond-level interval summaries.

# Recalculate the pond-level summary with accurate column names.
corrected_pond_gap_summary = (
    pond_dates.dropna(subset=["gap_days"])
    .groupby(["pond_id", "region"])
    .agg(
        number_of_intervals=("gap_days", "count"),
        mean_gap_days=("gap_days", "mean"),
        median_gap_days=("gap_days", "median"),
        minimum_gap_days=("gap_days", "min"),
        maximum_gap_days=("gap_days", "max"),
        gaps_1_to_3_days=("gap_days", lambda x: x.between(1, 3).sum()),
        gaps_over_30_days=("gap_days", lambda x: (x > 30).sum())
    )
    .reset_index()
)

# Count unique collection dates separately.
unique_dates_per_pond = (
    pond_dates.groupby("pond_id")["collection_date"]
    .nunique()
    .rename("unique_collection_dates")
    .reset_index()
)

corrected_pond_gap_summary = corrected_pond_gap_summary.merge(
    unique_dates_per_pond,
    on="pond_id",
    how="left"
)

# Save the corrected summary separately.
corrected_pond_gap_summary.to_csv(
    output_dir / "corrected_pond_level_monitoring_intervals.csv",
    index=False
)

print("AQUA — CORRECTED POND-LEVEL INTERVAL SUMMARY")
print(f"Ponds with calculated intervals: {len(corrected_pond_gap_summary):,}")
print(f"Total intervals: {corrected_pond_gap_summary['number_of_intervals'].sum():,}")

print("\nTOP 10 PONDS BY MEDIAN MONITORING GAP")
print(
    corrected_pond_gap_summary
    .sort_values("median_gap_days", ascending=False)
    .head(10)
    .round(2)
    .to_string(index=False)
)

print("\nCORRECTED SUMMARY SAVED:")
print(output_dir / "corrected_pond_level_monitoring_intervals.csv")

AQUA — CORRECTED POND-LEVEL INTERVAL SUMMARY
Ponds with calculated intervals: 304
Total intervals: 6,162

TOP 10 PONDS BY MEDIAN MONITORING GAP
      pond_id region  number_of_intervals  mean_gap_days  median_gap_days  minimum_gap_days  maximum_gap_days  gaps_1_to_3_days  gaps_over_30_days  unique_collection_dates
ara2_27387783  Eluru                    3          98.67           120.00             18.00            158.00                 0                  2                        4
ara2_1144b702  Eluru                    3          86.00            56.00             22.00            180.00                 0                  2                        4
ara2_1c72ee85  Eluru                    2          52.00            52.00             43.00             61.00                 0                  2                        3
ara2_87661414  Eluru                    7          64.14            49.00             10.00            164.00                 0                  4                      

In [25]:
# AQUA — Cell 27
# Name: follow_up_and_short_monitoring_interval_audit
# Purpose: Examine the association between short monitoring gaps and follow-up records.
# Related cells: 7, 8, 25 and 26.

import pandas as pd

# 1. Prepare a separate audit dataframe.
followup_df = df[
    ["pond_id", "region", "Date of data collection", "Is follow up"]
].copy()

followup_df["collection_date"] = pd.to_datetime(
    followup_df["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

followup_df["follow_up_flag"] = (
    followup_df["Is follow up"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("yes")
)

# 2. Aggregate records to one pond-date combination.
pond_date_followup = (
    followup_df.dropna(subset=["pond_id", "collection_date"])
    .groupby(["pond_id", "region", "collection_date"], as_index=False)
    .agg(
        records_on_date=("follow_up_flag", "size"),
        follow_up_records=("follow_up_flag", "sum")
    )
    .sort_values(["pond_id", "collection_date"])
)

# 3. Calculate the interval from the previous collection date.
pond_date_followup["gap_days"] = (
    pond_date_followup.groupby("pond_id")["collection_date"]
    .diff()
    .dt.days
)

# 4. Identify whether the later collection date contains a follow-up record.
pond_date_followup["has_follow_up"] = (
    pond_date_followup["follow_up_records"] > 0
)

# Keep only dates that have a preceding collection date.
interval_audit = pond_date_followup.dropna(subset=["gap_days"]).copy()

# 5. Categorise intervals descriptively.
interval_audit["gap_category"] = pd.cut(
    interval_audit["gap_days"],
    bins=[0, 3, 7, 30, 60, 90, float("inf")],
    labels=["1–3 days", "4–7 days", "8–30 days",
            "31–60 days", "61–90 days", "Over 90 days"]
)

# 6. Summarise follow-up association by interval category.
category_summary = (
    interval_audit.groupby("gap_category", observed=False)
    .agg(
        intervals=("gap_days", "count"),
        later_dates_with_follow_up=("has_follow_up", "sum"),
        median_gap_days=("gap_days", "median")
    )
)

category_summary["follow_up_percentage"] = (
    category_summary["later_dates_with_follow_up"]
    / category_summary["intervals"] * 100
).round(2)

# 7. Compare short intervals against all longer intervals.
interval_audit["interval_group"] = interval_audit["gap_days"].apply(
    lambda x: "1–3 days" if x <= 3 else "4+ days"
)

short_vs_long = (
    interval_audit.groupby("interval_group")
    .agg(
        intervals=("gap_days", "count"),
        later_dates_with_follow_up=("has_follow_up", "sum"),
        median_gap_days=("gap_days", "median")
    )
)

short_vs_long["follow_up_percentage"] = (
    short_vs_long["later_dates_with_follow_up"]
    / short_vs_long["intervals"] * 100
).round(2)

# 8. Count follow-up records separately from follow-up dates.
total_follow_up_records = int(
    pond_date_followup["follow_up_records"].sum()
)

total_follow_up_dates = int(
    pond_date_followup["has_follow_up"].sum()
)

short_interval_count = int(
    (interval_audit["gap_days"] <= 3).sum()
)

short_intervals_with_followup = int(
    (
        (interval_audit["gap_days"] <= 3)
        & interval_audit["has_follow_up"]
    ).sum()
)

# 9. Print one consolidated output block.
print("AQUA — FOLLOW-UP AND SHORT-INTERVAL AUDIT")
print(f"Unique pond-date combinations: {len(pond_date_followup):,}")
print(f"Consecutive-date intervals: {len(interval_audit):,}")
print(f"Collection dates containing follow-up records: {total_follow_up_dates:,}")
print(f"Total follow-up records: {total_follow_up_records:,}")

print("\nFOLLOW-UP ASSOCIATION BY INTERVAL CATEGORY")
print(category_summary.round(2).to_string())

print("\nSHORT INTERVALS (1–3 DAYS) VS LONGER INTERVALS")
print(short_vs_long.round(2).to_string())

print("\nSHORT-INTERVAL CHECK")
print(f"Short intervals: {short_interval_count:,}")
print(f"Short intervals followed by a date containing follow-up records: {short_intervals_with_followup:,}")

if short_interval_count:
    percentage = short_intervals_with_followup / short_interval_count * 100
    print(f"Percentage: {percentage:.2f}%")

print("\nINTERPRETATION NOTE")
print(
    "A follow-up record on the later collection date indicates an association, "
    "not proof that the short interval was caused by a follow-up visit."
)

AQUA — FOLLOW-UP AND SHORT-INTERVAL AUDIT
Unique pond-date combinations: 6,479
Consecutive-date intervals: 6,162
Collection dates containing follow-up records: 821
Total follow-up records: 1,570

FOLLOW-UP ASSOCIATION BY INTERVAL CATEGORY
              intervals  later_dates_with_follow_up  median_gap_days  follow_up_percentage
gap_category                                                                              
1–3 days            553                         542             2.00                 98.01
4–7 days            353                         246             5.00                 69.69
8–30 days          3646                          32            19.00                  0.88
31–60 days         1296                           0            36.00                  0.00
61–90 days          152                           1            71.00                  0.66
Over 90 days        162                           0           121.00                  0.00

SHORT INTERVALS (1–3 DAYS) VS LO

In [26]:
# AQUA — Cell 28
# Name: short_intervals_without_followup_investigation
# Purpose: Investigate short monitoring intervals not labelled as follow-up.
# Related cells: 25, 26 and 27.

# 1. Reconstruct consecutive collection dates and their preceding dates.
cell28_dates = pond_date_followup.copy()

cell28_dates["previous_collection_date"] = (
    cell28_dates.groupby("pond_id")["collection_date"].shift(1)
)

cell28_dates["gap_days"] = (
    cell28_dates["collection_date"]
    - cell28_dates["previous_collection_date"]
).dt.days

# 2. Identify short intervals whose later date has no follow-up record.
exceptions = cell28_dates[
    cell28_dates["gap_days"].between(1, 3)
    & ~cell28_dates["has_follow_up"]
].copy()

# 3. Retrieve original measurements from the later collection dates.
later_records = df.copy()

later_records["collection_date"] = pd.to_datetime(
    later_records["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

exception_details = exceptions[
    [
        "pond_id",
        "region",
        "previous_collection_date",
        "collection_date",
        "gap_days",
        "records_on_date",
        "follow_up_records"
    ]
].merge(
    later_records,
    on=["pond_id", "region", "collection_date"],
    how="left"
)

# 4. Summarise measurement Type and target for the later dates.
type_summary = (
    exception_details["Type"]
    .fillna("Missing")
    .value_counts()
)

target_summary = (
    exception_details["Is WQ in range?"]
    .fillna("Missing")
    .value_counts()
)

# 5. Prepare a compact record-level investigation table.
columns_to_show = [
    "pond_id",
    "region",
    "previous_collection_date",
    "collection_date",
    "gap_days",
    "Type",
    "Is follow up",
    "Is WQ in range?"
]

record_details = exception_details[columns_to_show].copy()

record_details["previous_collection_date"] = (
    record_details["previous_collection_date"].dt.strftime("%Y-%m-%d")
)

record_details["collection_date"] = (
    record_details["collection_date"].dt.strftime("%Y-%m-%d")
)

# 6. Print one consolidated output block.
print("AQUA — SHORT INTERVALS WITHOUT FOLLOW-UP LABELS")
print(f"Short intervals without a follow-up label: {len(exceptions):,}")
print(f"Original measurement records on later dates: {len(exception_details):,}")

print("\nMEASUREMENT TYPE COUNTS")
print(type_summary.to_string())

print("\nWATER-QUALITY TARGET COUNTS")
print(target_summary.to_string())

print("\nEXCEPTION RECORD DETAILS")
print(record_details.to_string(index=False))

AQUA — SHORT INTERVALS WITHOUT FOLLOW-UP LABELS
Short intervals without a follow-up label: 11
Original measurement records on later dates: 19

MEASUREMENT TYPE COUNTS
Type
Morning    10
Evening     9

WATER-QUALITY TARGET COUNTS
Is WQ in range?
Yes    12
No      7

EXCEPTION RECORD DETAILS
      pond_id  region previous_collection_date collection_date  gap_days    Type Is follow up Is WQ in range?
ara2_01463d19   Eluru               2022-11-14      2022-11-16      2.00 Evening           No             Yes
ara2_01463d19   Eluru               2022-11-14      2022-11-16      2.00 Morning           No             Yes
ara2_01463d19   Eluru               2023-04-18      2023-04-21      3.00 Evening           No              No
ara2_01463d19   Eluru               2023-04-18      2023-04-21      3.00 Morning           No              No
ara2_51ee6ab1 Nellore               2022-01-07      2022-01-10      3.00 Evening           No              No
ara2_51ee6ab1 Nellore               2022-01-07   

In [27]:
# AQUA — Cell 31
# RE-EDITED VERSION OF CELL 29,30
# Name: previous_date_water_quality_status_investigation
# Purpose: Compare previous and later water-quality status for short intervals without follow-up labels.

# 1. Retrieve the 11 exception intervals identified in Cell 28.
cell30_intervals = exceptions[
    [
        "pond_id",
        "region",
        "previous_collection_date",
        "collection_date",
        "gap_days"
    ]
].copy()

# 2. Prepare original measurements.
cell30_records = df.copy()

cell30_records["collection_date"] = pd.to_datetime(
    cell30_records["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

# 3. Prepare previous-date records.
# Rename the date column before merging to avoid a column-name collision.
previous_source = cell30_records.rename(
    columns={"collection_date": "previous_collection_date"}
)

previous_records = cell30_intervals.merge(
    previous_source,
    on=["pond_id", "region", "previous_collection_date"],
    how="left"
)

# 4. Prepare later-date records.
later_records = cell30_intervals.merge(
    cell30_records,
    on=["pond_id", "region", "collection_date"],
    how="left"
)

# 5. Summarise previous-date target status.
group_columns = [
    "pond_id",
    "region",
    "previous_collection_date",
    "collection_date",
    "gap_days"
]

previous_status = (
    previous_records.groupby(group_columns, dropna=False)
    .agg(
        previous_measurement_records=("Is WQ in range?", "size"),
        previous_target_no=("Is WQ in range?", lambda x: (x == "No").sum()),
        previous_target_yes=("Is WQ in range?", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

# 6. Summarise later-date target status.
later_status = (
    later_records.groupby(group_columns, dropna=False)
    .agg(
        later_measurement_records=("Is WQ in range?", "size"),
        later_target_no=("Is WQ in range?", lambda x: (x == "No").sum()),
        later_target_yes=("Is WQ in range?", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

# 7. Combine previous and later date summaries.
comparison = previous_status.merge(
    later_status,
    on=group_columns,
    how="inner"
)

# 8. Count intervals preceded by at least one target No.
previous_date_with_no = int(
    (comparison["previous_target_no"] > 0).sum()
)

previous_date_without_no = int(
    (comparison["previous_target_no"] == 0).sum()
)

# 9. Print one consolidated output block.
print("AQUA — PREVIOUS-DATE WATER-QUALITY STATUS INVESTIGATION")
print(f"Exception intervals investigated: {len(comparison):,}")
print(f"Intervals preceded by at least one target No: {previous_date_with_no:,}")
print(f"Intervals without a target No on the previous date: {previous_date_without_no:,}")

print("\nPREVIOUS VS LATER COLLECTION-DATE STATUS")
print(
    comparison.sort_values(
        ["pond_id", "previous_collection_date"]
    ).to_string(index=False)
)

print("\nINTERPRETATION NOTE")
print(
    "A previous-date target No indicates an out-of-range classification "
    "on that date. It does not establish why the later visit occurred."
)

AQUA — PREVIOUS-DATE WATER-QUALITY STATUS INVESTIGATION
Exception intervals investigated: 11
Intervals preceded by at least one target No: 1
Intervals without a target No on the previous date: 10

PREVIOUS VS LATER COLLECTION-DATE STATUS
      pond_id  region previous_collection_date collection_date  gap_days  previous_measurement_records  previous_target_no  previous_target_yes  later_measurement_records  later_target_no  later_target_yes
ara2_01463d19   Eluru               2022-11-14      2022-11-16      2.00                             2                   0                    2                          2                0                 2
ara2_01463d19   Eluru               2023-04-18      2023-04-21      3.00                             2                   0                    2                          2                2                 0
ara2_51ee6ab1 Nellore               2022-01-07      2022-01-10      3.00                             1                   0                    1 

In [28]:
# AQUA — Cell 33
# RE-EDITED VERSION OF CELL 32
# Name: exception_interval_followup_intervention_investigation
# Purpose: Inspect follow-up and intervention fields on both dates
# for the 11 short monitoring intervals without later follow-up labels.

import pandas as pd

# 1. Retrieve the 11 exception intervals from Cell 28.
cell33_intervals = exceptions[
    [
        "pond_id",
        "region",
        "previous_collection_date",
        "collection_date",
        "gap_days"
    ]
].copy()

# 2. Prepare a copy of the original dataset.
cell33_records = df.copy()

cell33_records["collection_date"] = pd.to_datetime(
    cell33_records["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

# 3. Identify available follow-up and intervention-related columns.
keywords = [
    "follow up",
    "helped",
    "corrective",
    "implement",
    "improved",
    "self-initiated",
    "reason",
    "action"
]

cell33_relevant_columns = [
    column
    for column in cell33_records.columns
    if any(keyword in column.lower() for keyword in keywords)
]

# 4. Include only identifier columns that actually exist.
cell33_preferred_identifiers = [
    "pond_id",
    "region",
    "Date of data collection",
    "collection_date",
    "Time",
    "Type",
    "Is WQ in range?"
]

cell33_identifier_columns = [
    column
    for column in cell33_preferred_identifiers
    if column in cell33_records.columns
]

cell33_selected_columns = list(dict.fromkeys(
    cell33_identifier_columns + cell33_relevant_columns
))

# 5. Retrieve records from both dates for every exception interval.
cell33_results = []

for _, interval in cell33_intervals.iterrows():

    for date_role, target_date in [
        ("Previous date", interval["previous_collection_date"]),
        ("Later date", interval["collection_date"])
    ]:

        matched = cell33_records[
            (cell33_records["pond_id"] == interval["pond_id"]) &
            (cell33_records["region"] == interval["region"]) &
            (cell33_records["collection_date"] == target_date)
        ][cell33_selected_columns].copy()

        matched.insert(0, "date_role", date_role)
        matched.insert(0, "gap_days", interval["gap_days"])
        matched.insert(0, "later_collection_date", interval["collection_date"])
        matched.insert(0, "previous_collection_date", interval["previous_collection_date"])

        cell33_results.append(matched)

# 6. Combine the retrieved records.
cell33_investigation = pd.concat(
    cell33_results,
    ignore_index=True
)

# 7. Print one consolidated output block.
print("AQUA — EXCEPTION INTERVAL FOLLOW-UP AND INTERVENTION INVESTIGATION")
print(f"Exception intervals investigated: {len(cell33_intervals):,}")
print(f"Relevant fields identified: {len(cell33_relevant_columns):,}")
print(f"Measurement records retrieved: {len(cell33_investigation):,}")

print("\nAVAILABLE IDENTIFIER FIELDS")
print(", ".join(cell33_identifier_columns))

print("\nFOLLOW-UP AND INTERVENTION FIELDS")
print(", ".join(cell33_relevant_columns))

print("\nPREVIOUS-DATE AND LATER-DATE RECORD DETAILS")
print(
    cell33_investigation
    .sort_values(
        ["pond_id", "previous_collection_date", "date_role"],
        na_position="last"
    )
    .to_string(index=False)
)

AQUA — EXCEPTION INTERVAL FOLLOW-UP AND INTERVENTION INVESTIGATION
Exception intervals investigated: 11
Relevant fields identified: 21
Measurement records retrieved: 36

AVAILABLE IDENTIFIER FIELDS
pond_id, region, Date of data collection, collection_date, Type, Is WQ in range?

FOLLOW-UP AND INTERVENTION FIELDS
Is follow up, Is follow up possible, Reason follow up not possible, Corrective actions requested, Corrective actions requested (other), Corrective actions amount requested, Corrective actions implemented, Corrective actions implementation date, Corrective actions taken, Corrective actions taken (other), Corrective actions taken (details), Non-prescribed corrective actions taken, Reason not implemented, Water quality improved after corrective actions, Corrective action notes, Self-initiated corrective actions taken, Self-initiated corrective actions implemented on (exact date), Self-initiated corrective actions implemented (date range), Self-initiated corrective actions notes, H

In [29]:
# AQUA — Cell 34
# Name: exception_interval_level_followup_summary
# Purpose: Summarise follow-up and intervention indicators at the interval level.

# 1. Start with the detailed records retrieved in Cell 33.
cell34_data = cell33_investigation.copy()

# 2. Define the columns used for interval identification.
cell34_interval_keys = [
    "pond_id",
    "region",
    "previous_collection_date",
    "later_collection_date",
    "gap_days"
]

# 3. Summarise each interval separately.
cell34_summary = (
    cell34_data.groupby(cell34_interval_keys, dropna=False)
    .apply(
        lambda group: pd.Series({
            "previous_followup_yes": (
                group.loc[
                    group["date_role"] == "Previous date",
                    "Is follow up"
                ].eq("Yes").any()
            ),
            "later_followup_yes": (
                group.loc[
                    group["date_role"] == "Later date",
                    "Is follow up"
                ].eq("Yes").any()
            ),
            "previous_corrective_action_recorded": (
                group.loc[
                    group["date_role"] == "Previous date",
                    "Corrective actions taken"
                ].notna().any()
            ),
            "previous_self_initiated_action_recorded": (
                group.loc[
                    group["date_role"] == "Previous date",
                    "Self-initiated corrective actions taken"
                ].notna().any()
            ),
            "previous_target_no": (
                group.loc[
                    group["date_role"] == "Previous date",
                    "Is WQ in range?"
                ].eq("No").any()
            ),
            "later_target_no": (
                group.loc[
                    group["date_role"] == "Later date",
                    "Is WQ in range?"
                ].eq("No").any()
            )
        })
    )
    .reset_index()
)

# 4. Count interval-level patterns.
print("AQUA — INTERVAL-LEVEL FOLLOW-UP SUMMARY")
print(f"Total exception intervals: {len(cell34_summary):,}")

print(
    f"Previous dates with follow-up label Yes: "
    f"{cell34_summary['previous_followup_yes'].sum():,}"
)

print(
    f"Later dates with follow-up label Yes: "
    f"{cell34_summary['later_followup_yes'].sum():,}"
)

print(
    f"Previous dates with corrective-action records: "
    f"{cell34_summary['previous_corrective_action_recorded'].sum():,}"
)

print(
    f"Previous dates with self-initiated action records: "
    f"{cell34_summary['previous_self_initiated_action_recorded'].sum():,}"
)

print(
    f"Previous dates containing target No: "
    f"{cell34_summary['previous_target_no'].sum():,}"
)

print(
    f"Later dates containing target No: "
    f"{cell34_summary['later_target_no'].sum():,}"
)

print("\nINTERVAL-LEVEL DETAILS")
print(
    cell34_summary.sort_values(
        ["pond_id", "previous_collection_date"]
    ).to_string(index=False)
)

AQUA — INTERVAL-LEVEL FOLLOW-UP SUMMARY
Total exception intervals: 11
Previous dates with follow-up label Yes: 5
Later dates with follow-up label Yes: 0
Previous dates with corrective-action records: 5
Previous dates with self-initiated action records: 0
Previous dates containing target No: 1
Later dates containing target No: 4

INTERVAL-LEVEL DETAILS
      pond_id  region previous_collection_date later_collection_date  gap_days  previous_followup_yes  later_followup_yes  previous_corrective_action_recorded  previous_self_initiated_action_recorded  previous_target_no  later_target_no
ara2_01463d19   Eluru               2022-11-14            2022-11-16      2.00                   True               False                                 True                                    False               False            False
ara2_01463d19   Eluru               2023-04-18            2023-04-21      3.00                   True               False                                 True              

In [30]:
# AQUA — Cell 35
# Name: exception_interval_measurement_context_investigation
# Purpose: Compare measurements and visit context across the 11 exception intervals.

import pandas as pd

# 1. Retrieve the 11 exception intervals.
cell35_intervals = exceptions[
    [
        "pond_id",
        "region",
        "previous_collection_date",
        "collection_date",
        "gap_days"
    ]
].copy()

# 2. Prepare a copy of the original dataset.
cell35_records = df.copy()

cell35_records["collection_date"] = pd.to_datetime(
    cell35_records["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

# 3. Define the fields to investigate.
# Only fields actually present in the dataset will be selected.
cell35_candidate_columns = [
    "Type",
    "Is WQ in range?",
    "DO",
    "pH",
    "Turbidity",
    "TDS",
    "Temp",
    "Ammonia (TAN-N)",
    "Ammonia (TAN) (NH3)",
    "Ammonia (NH3)",
    "Ammonia—TAN (NH3-N)",
    "Ammonia—TAN (NH3)",
    "Ammonia—NH3",
    "Observer",
    "Equipment",
    "Weather",
    "Pond status",
    "Group",
    "Species",
    "Feed amount",
    "Dead fish",
    "Individuals air gulping",
    "Individuals tail splashing",
    "Water color",
    "Notes"
]

cell35_available_columns = [
    column
    for column in cell35_candidate_columns
    if column in cell35_records.columns
]

cell35_missing_columns = [
    column
    for column in cell35_candidate_columns
    if column not in cell35_records.columns
]

# 4. Retrieve records from both dates for every exception interval.
cell35_results = []

for _, interval in cell35_intervals.iterrows():

    for date_role, target_date in [
        ("Previous date", interval["previous_collection_date"]),
        ("Later date", interval["collection_date"])
    ]:

        matched = cell35_records[
            (cell35_records["pond_id"] == interval["pond_id"]) &
            (cell35_records["region"] == interval["region"]) &
            (cell35_records["collection_date"] == target_date)
        ][cell35_available_columns].copy()

        matched.insert(0, "date_role", date_role)
        matched.insert(0, "gap_days", interval["gap_days"])
        matched.insert(0, "later_collection_date", interval["collection_date"])
        matched.insert(0, "previous_collection_date", interval["previous_collection_date"])
        matched.insert(0, "pond_id", interval["pond_id"])
        matched.insert(0, "region", interval["region"])

        cell35_results.append(matched)

# 5. Combine retrieved records.
cell35_investigation = pd.concat(
    cell35_results,
    ignore_index=True
)

# 6. Print one consolidated output block.
print("AQUA — EXCEPTION INTERVAL MEASUREMENT AND VISIT-CONTEXT INVESTIGATION")

print(f"Exception intervals investigated: {len(cell35_intervals):,}")
print(f"Measurement records retrieved: {len(cell35_investigation):,}")
print(f"Candidate fields available: {len(cell35_available_columns):,}")
print(f"Candidate fields not found: {len(cell35_missing_columns):,}")

print("\nAVAILABLE FIELDS")
print(", ".join(cell35_available_columns))

print("\nCANDIDATE FIELDS NOT FOUND")
print(", ".join(cell35_missing_columns) if cell35_missing_columns else "None")

print("\nPREVIOUS-DATE AND LATER-DATE MEASUREMENTS")
print(
    cell35_investigation
    .sort_values(
        ["pond_id", "previous_collection_date", "date_role", "Type"],
        na_position="last"
    )
    .to_string(index=False)
)

AQUA — EXCEPTION INTERVAL MEASUREMENT AND VISIT-CONTEXT INVESTIGATION
Exception intervals investigated: 11
Measurement records retrieved: 36
Candidate fields available: 14
Candidate fields not found: 11

AVAILABLE FIELDS
Type, Is WQ in range?, pH, Observer, Equipment, Weather, Pond status, Group, Species, Dead fish, Individuals air gulping, Individuals tail splashing, Water color, Notes

CANDIDATE FIELDS NOT FOUND
DO, Turbidity, TDS, Temp, Ammonia (TAN-N), Ammonia (TAN) (NH3), Ammonia (NH3), Ammonia—TAN (NH3-N), Ammonia—TAN (NH3), Ammonia—NH3, Feed amount

PREVIOUS-DATE AND LATER-DATE MEASUREMENTS
 region       pond_id previous_collection_date later_collection_date  gap_days     date_role    Type Is WQ in range?   pH    Observer                                                 Equipment      Weather Pond status            Group                                  Species  Dead fish  Individuals air gulping  Individuals tail splashing Water color                                             

In [31]:
# AQUA — Cell 36
# Name: actual_measurement_column_name_verification
# Purpose: Identify actual measurement-related column names in the original dataset.
# This cell verifies names before any further measurement analysis.

cell36_columns = list(df.columns)

cell36_keywords = [
    "do",
    "dissolved oxygen",
    "oxygen",
    "ph",
    "turbidity",
    "tds",
    "temperature",
    "temp",
    "ammonia",
    "tan",
    "feed",
    "alkalinity",
    "hardness"
]

cell36_matches = {}

for keyword in cell36_keywords:
    matches = [
        column
        for column in cell36_columns
        if keyword.lower() in column.lower()
    ]

    if matches:
        cell36_matches[keyword] = matches

print("AQUA — ACTUAL MEASUREMENT COLUMN-NAME VERIFICATION")
print(f"Total dataset columns: {len(cell36_columns)}")
print(f"Measurement-related keyword groups found: {len(cell36_matches)}")

print("\nEXACT MATCHING COLUMN NAMES")

for keyword, matches in cell36_matches.items():
    print(f"{keyword}: {matches}")

print("\nCOMPLETE ORIGINAL COLUMN INVENTORY")

for number, column in enumerate(cell36_columns, start=1):
    print(f"{number}. {column}")

AQUA — ACTUAL MEASUREMENT COLUMN-NAME VERIFICATION
Total dataset columns: 53
Measurement-related keyword groups found: 10

EXACT MATCHING COLUMN NAMES
do: ['DO (mg/L)']
ph: ['pH']
turbidity: ['Turbidity (in cm)']
tds: ['TDS (ppt)']
temp: ['Temp (in °C)']
ammonia: ['Ammonia—TAN (NH3-N) (mg/L)', 'Ammonia—TAN (NH3) (mg/L)', 'Ammonia—NH3 (mg/L)']
tan: ['Ammonia—TAN (NH3-N) (mg/L)', 'Ammonia—TAN (NH3) (mg/L)']
feed: ['Feed amount (kg)']
alkalinity: ['Alkalinity (mg/L)']
hardness: ['Hardness (mg/L)']

COMPLETE ORIGINAL COLUMN INVENTORY
1. pond_id
2. region
3. Date of data collection
4. Time of data collection
5. Type
6. Is follow up
7. Is follow up possible
8. Reason follow up not possible
9. Group
10. Pond status
11. Observer
12. Equipment
13. Weather
14. DO (mg/L)
15. pH
16. Turbidity (in cm)
17. Ammonia—TAN (NH3-N) (mg/L)
18. Ammonia—TAN (NH3) (mg/L)
19. Ammonia—NH3 (mg/L)
20. Temp (in °C)
21. TDS (ppt)
22. Alkalinity (mg/L)
23. Hardness (mg/L)
24. Water color
25. Is WQ in range?
26. Para

In [32]:
# AQUA — Cell 37
# RE-EDITED VERSION OF CELL 35
# Name: verified_exception_interval_measurement_context_investigation
# Purpose: Compare verified measurements and visit context across both dates.

import pandas as pd

# 1. Retrieve the 11 exception intervals.
cell37_intervals = exceptions[
    [
        "pond_id",
        "region",
        "previous_collection_date",
        "collection_date",
        "gap_days"
    ]
].copy()

# 2. Prepare a copy of the original dataset.
cell37_records = df.copy()

cell37_records["collection_date"] = pd.to_datetime(
    cell37_records["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

# 3. Use exact column names verified in Cell 36.
cell37_candidate_columns = [
    "Time of data collection",
    "Type",
    "Is WQ in range?",
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "TDS (ppt)",
    "Temp (in °C)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Feed amount (kg)",
    "Stocking density (per acre)",
    "Weight",
    "Observer",
    "Equipment",
    "Weather",
    "Pond status",
    "Group",
    "Species",
    "Dead fish",
    "Individuals air gulping",
    "Individuals tail splashing",
    "Water color",
    "Notes"
]

# 4. Verify that every requested field exists before proceeding.
cell37_missing_columns = [
    column
    for column in cell37_candidate_columns
    if column not in cell37_records.columns
]

if cell37_missing_columns:
    raise KeyError(
        f"Verified candidate columns unexpectedly missing: "
        f"{cell37_missing_columns}"
    )

# 5. Retrieve records from both dates for every exception interval.
cell37_results = []

for _, interval in cell37_intervals.iterrows():

    for date_role, target_date in [
        ("Previous date", interval["previous_collection_date"]),
        ("Later date", interval["collection_date"])
    ]:

        matched = cell37_records[
            (cell37_records["pond_id"] == interval["pond_id"]) &
            (cell37_records["region"] == interval["region"]) &
            (cell37_records["collection_date"] == target_date)
        ][cell37_candidate_columns].copy()

        matched.insert(0, "date_role", date_role)
        matched.insert(0, "gap_days", interval["gap_days"])
        matched.insert(0, "later_collection_date", interval["collection_date"])
        matched.insert(0, "previous_collection_date", interval["previous_collection_date"])
        matched.insert(0, "pond_id", interval["pond_id"])
        matched.insert(0, "region", interval["region"])

        cell37_results.append(matched)

# 6. Combine the retrieved records.
cell37_investigation = pd.concat(
    cell37_results,
    ignore_index=True
)

# 7. Verify interval and record counts.
cell37_retrieved_intervals = (
    cell37_investigation[
        ["pond_id", "region", "previous_collection_date", "later_collection_date"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("AQUA — VERIFIED EXCEPTION INTERVAL MEASUREMENT INVESTIGATION")
print(f"Exception intervals requested: {len(cell37_intervals):,}")
print(f"Distinct intervals retrieved: {cell37_retrieved_intervals:,}")
print(f"Measurement records retrieved: {len(cell37_investigation):,}")
print(f"Verified fields selected: {len(cell37_candidate_columns):,}")
print(f"Missing requested fields: {len(cell37_missing_columns):,}")

print("\nPREVIOUS-DATE AND LATER-DATE RECORDS")
print(
    cell37_investigation
    .sort_values(
        [
            "pond_id",
            "previous_collection_date",
            "date_role",
            "Time of data collection"
        ],
        na_position="last"
    )
    .to_string(index=False)
)

AQUA — VERIFIED EXCEPTION INTERVAL MEASUREMENT INVESTIGATION
Exception intervals requested: 11
Distinct intervals retrieved: 11
Measurement records retrieved: 36
Verified fields selected: 25
Missing requested fields: 0

PREVIOUS-DATE AND LATER-DATE RECORDS
 region       pond_id previous_collection_date later_collection_date  gap_days     date_role Time of data collection    Type Is WQ in range?  DO (mg/L)   pH  Turbidity (in cm)  TDS (ppt)  Temp (in °C)  Ammonia—TAN (NH3-N) (mg/L)  Ammonia—TAN (NH3) (mg/L)  Ammonia—NH3 (mg/L)  Feed amount (kg)  Stocking density (per acre)  Weight    Observer                                                 Equipment      Weather Pond status            Group                                  Species  Dead fish  Individuals air gulping  Individuals tail splashing Water color                                                                                                                                                                                         

In [33]:
# AQUA — Cell 38
# Name: same_type_exception_interval_comparison
# Purpose: Compare previous and later measurements within the same observation Type.

cell38_group_columns = [
    "region",
    "pond_id",
    "previous_collection_date",
    "later_collection_date",
    "gap_days",
    "Type"
]

cell38_measurement_columns = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "TDS (ppt)",
    "Temp (in °C)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)"
]

cell38_results = []

for keys, group in cell37_investigation.groupby(
    cell38_group_columns,
    dropna=False
):

    previous = group[group["date_role"] == "Previous date"]
    later = group[group["date_role"] == "Later date"]

    result = dict(zip(cell38_group_columns, keys))

    result["previous_records"] = len(previous)
    result["later_records"] = len(later)

    for column in ["Is WQ in range?"] + cell38_measurement_columns:

        previous_values = previous[column].dropna().unique()
        later_values = later[column].dropna().unique()

        result[f"previous_{column}"] = list(previous_values)
        result[f"later_{column}"] = list(later_values)

    cell38_results.append(result)

cell38_comparison = pd.DataFrame(cell38_results)

print("AQUA — SAME-TYPE EXCEPTION INTERVAL COMPARISON")
print(f"Comparison groups: {len(cell38_comparison):,}")
print(f"Groups with previous records: {(cell38_comparison['previous_records'] > 0).sum():,}")
print(f"Groups with later records: {(cell38_comparison['later_records'] > 0).sum():,}")

print("\nCOMPARISON RESULTS")
print(cell38_comparison.to_string(index=False))

AQUA — SAME-TYPE EXCEPTION INTERVAL COMPARISON
Comparison groups: 22
Groups with previous records: 17
Groups with later records: 19

COMPARISON RESULTS
 region       pond_id previous_collection_date later_collection_date  gap_days    Type  previous_records  later_records previous_Is WQ in range? later_Is WQ in range? previous_DO (mg/L) later_DO (mg/L) previous_pH later_pH previous_Turbidity (in cm) later_Turbidity (in cm) previous_TDS (ppt) later_TDS (ppt) previous_Temp (in °C) later_Temp (in °C) previous_Ammonia—TAN (NH3-N) (mg/L) later_Ammonia—TAN (NH3-N) (mg/L) previous_Ammonia—TAN (NH3) (mg/L) later_Ammonia—TAN (NH3) (mg/L) previous_Ammonia—NH3 (mg/L) later_Ammonia—NH3 (mg/L)
  Eluru ara2_01463d19               2022-11-14            2022-11-16      2.00 Evening                 1              1                    [Yes]                 [Yes]              [8.8]          [12.0]      [8.28]   [8.14]                         []                      []                 []              []   

In [34]:
# AQUA — Cell 39
# Name: matched_same_type_measurement_change_audit
# Purpose: Calculate between-visit changes only for genuinely matched records.

cell39_group_columns = [
    "region",
    "pond_id",
    "previous_collection_date",
    "later_collection_date",
    "gap_days",
    "Type"
]

cell39_measurements = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "TDS (ppt)",
    "Temp (in °C)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)"
]

cell39_matched = []
cell39_unmatched = []

for keys, group in cell37_investigation.groupby(
    cell39_group_columns,
    dropna=False
):

    previous = group[group["date_role"] == "Previous date"]
    later = group[group["date_role"] == "Later date"]

    if len(previous) != 1 or len(later) != 1:
        cell39_unmatched.append({
            **dict(zip(cell39_group_columns, keys)),
            "previous_records": len(previous),
            "later_records": len(later)
        })
        continue

    previous = previous.iloc[0]
    later = later.iloc[0]

    result = dict(zip(cell39_group_columns, keys))

    result["previous_target"] = previous["Is WQ in range?"]
    result["later_target"] = later["Is WQ in range?"]

    for column in cell39_measurements:

        previous_value = pd.to_numeric(
            pd.Series([previous[column]]),
            errors="coerce"
        ).iloc[0]

        later_value = pd.to_numeric(
            pd.Series([later[column]]),
            errors="coerce"
        ).iloc[0]

        result[f"{column}_change"] = (
            later_value - previous_value
            if pd.notna(previous_value) and pd.notna(later_value)
            else float("nan")
        )

    cell39_matched.append(result)

cell39_matched_df = pd.DataFrame(cell39_matched)
cell39_unmatched_df = pd.DataFrame(cell39_unmatched)

print("AQUA — MATCHED SAME-TYPE MEASUREMENT CHANGE AUDIT")
print(f"Total pond-interval-Type groups: {len(cell39_matched) + len(cell39_unmatched)}")
print(f"Groups with exactly one record on both dates: {len(cell39_matched)}")
print(f"Groups excluded from change calculations: {len(cell39_unmatched)}")

if not cell39_matched_df.empty:

    print("\nTARGET TRANSITIONS")

    print(
        cell39_matched_df
        .groupby(
            ["previous_target", "later_target"],
            dropna=False
        )
        .size()
        .rename("group_count")
        .to_string()
    )

    print("\nAVAILABLE MEASUREMENT CHANGE COUNTS")

    for column in cell39_measurements:
        change_column = f"{column}_change"
        available = cell39_matched_df[change_column].notna().sum()
        print(f"{column}: {available} matched changes")

    print("\nMATCHED MEASUREMENT CHANGES")

    print(
        cell39_matched_df
        .to_string(index=False)
    )

print("\nUNMATCHED GROUPS")

if cell39_unmatched_df.empty:
    print("None")
else:
    print(cell39_unmatched_df.to_string(index=False))

AQUA — MATCHED SAME-TYPE MEASUREMENT CHANGE AUDIT
Total pond-interval-Type groups: 22
Groups with exactly one record on both dates: 14
Groups excluded from change calculations: 8

TARGET TRANSITIONS
previous_target  later_target
No               Yes             2
Yes              No              5
                 Yes             7

AVAILABLE MEASUREMENT CHANGE COUNTS
DO (mg/L): 13 matched changes
pH: 13 matched changes
Turbidity (in cm): 8 matched changes
TDS (ppt): 5 matched changes
Temp (in °C): 14 matched changes
Ammonia—TAN (NH3-N) (mg/L): 5 matched changes
Ammonia—TAN (NH3) (mg/L): 0 matched changes
Ammonia—NH3 (mg/L): 0 matched changes

MATCHED MEASUREMENT CHANGES
 region       pond_id previous_collection_date later_collection_date  gap_days    Type previous_target later_target  DO (mg/L)_change  pH_change  Turbidity (in cm)_change  TDS (ppt)_change  Temp (in °C)_change  Ammonia—TAN (NH3-N) (mg/L)_change  Ammonia—TAN (NH3) (mg/L)_change  Ammonia—NH3 (mg/L)_change
  Eluru ara2_01

In [35]:
# AQUA — Cell 40
# Name: dataset_wide_target_distribution_audit
# Purpose: Examine target distribution across years, regions and observation types.
# Original dataframe remains unchanged.

import pandas as pd

# 1. Create an independent working copy.
cell40_data = df[
    [
        "Date of data collection",
        "region",
        "Type",
        "Is WQ in range?"
    ]
].copy()

# 2. Parse dates using the verified original date format.
cell40_data["collection_date"] = pd.to_datetime(
    cell40_data["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

cell40_data["year"] = cell40_data["collection_date"].dt.year

# 3. Verify target labels and missing values.
print("AQUA — DATASET-WIDE TARGET DISTRIBUTION AUDIT")
print(f"Total observations: {len(cell40_data):,}")
print(f"Unparseable dates: {cell40_data['collection_date'].isna().sum():,}")
print(f"Missing target values: {cell40_data['Is WQ in range?'].isna().sum():,}")

print("\nOVERALL TARGET DISTRIBUTION")

print(
    cell40_data["Is WQ in range?"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nTARGET DISTRIBUTION BY YEAR — COUNTS")

cell40_year_counts = pd.crosstab(
    cell40_data["year"],
    cell40_data["Is WQ in range?"],
    dropna=False
)

print(cell40_year_counts.to_string())

print("\nTARGET DISTRIBUTION BY YEAR — ROW PERCENTAGES")

cell40_year_percentages = (
    pd.crosstab(
        cell40_data["year"],
        cell40_data["Is WQ in range?"],
        normalize="index"
    ) * 100
)

print(cell40_year_percentages.round(2).to_string())

print("\nTARGET DISTRIBUTION BY REGION — COUNTS")

cell40_region_counts = pd.crosstab(
    cell40_data["region"],
    cell40_data["Is WQ in range?"],
    dropna=False
)

print(cell40_region_counts.to_string())

print("\nTARGET DISTRIBUTION BY REGION — ROW PERCENTAGES")

cell40_region_percentages = (
    pd.crosstab(
        cell40_data["region"],
        cell40_data["Is WQ in range?"],
        normalize="index"
    ) * 100
)

print(cell40_region_percentages.round(2).to_string())

print("\nTARGET DISTRIBUTION BY OBSERVATION TYPE — COUNTS")

cell40_type_counts = pd.crosstab(
    cell40_data["Type"],
    cell40_data["Is WQ in range?"],
    dropna=False
)

print(cell40_type_counts.to_string())

print("\nTARGET DISTRIBUTION BY OBSERVATION TYPE — ROW PERCENTAGES")

cell40_type_percentages = (
    pd.crosstab(
        cell40_data["Type"],
        cell40_data["Is WQ in range?"],
        normalize="index"
    ) * 100
)

print(cell40_type_percentages.round(2).to_string())

AQUA — DATASET-WIDE TARGET DISTRIBUTION AUDIT
Total observations: 12,431
Unparseable dates: 0
Missing target values: 0

OVERALL TARGET DISTRIBUTION
Is WQ in range?
Yes    10064
No      2367

TARGET DISTRIBUTION BY YEAR — COUNTS
Is WQ in range?   No   Yes
year                      
2021             418   252
2022             726  1486
2023             430  1661
2024             266  2227
2025             243  2761
2026             284  1677

TARGET DISTRIBUTION BY YEAR — ROW PERCENTAGES
Is WQ in range?    No   Yes
year                       
2021            62.39 37.61
2022            32.82 67.18
2023            20.56 79.44
2024            10.67 89.33
2025             8.09 91.91
2026            14.48 85.52

TARGET DISTRIBUTION BY REGION — COUNTS
Is WQ in range?    No   Yes
region                     
Eluru            1699  7549
Nellore           668  2515

TARGET DISTRIBUTION BY REGION — ROW PERCENTAGES
Is WQ in range?    No   Yes
region                     
Eluru           18.37 81.63


In [36]:
# AQUA — Cell 41
# Name: yearly_regional_target_distribution_audit
# Purpose: Examine target distribution by year within each region.
# Original dataframe remains unchanged.

cell41_data = df[
    [
        "Date of data collection",
        "region",
        "Is WQ in range?"
    ]
].copy()

cell41_data["collection_date"] = pd.to_datetime(
    cell41_data["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

cell41_data["year"] = cell41_data["collection_date"].dt.year

print("AQUA — YEARLY REGIONAL TARGET DISTRIBUTION AUDIT")
print(f"Total observations: {len(cell41_data):,}")
print(f"Unparseable dates: {cell41_data['collection_date'].isna().sum():,}")
print(f"Missing region values: {cell41_data['region'].isna().sum():,}")
print(f"Missing target values: {cell41_data['Is WQ in range?'].isna().sum():,}")

print("\nOBSERVATION COUNTS BY YEAR AND REGION")

cell41_counts = pd.crosstab(
    [cell41_data["year"], cell41_data["region"]],
    cell41_data["Is WQ in range?"]
)

print(cell41_counts.to_string())

print("\nTARGET PERCENTAGES WITHIN EACH YEAR-REGION GROUP")

cell41_percentages = (
    pd.crosstab(
        [cell41_data["year"], cell41_data["region"]],
        cell41_data["Is WQ in range?"],
        normalize="index"
    ) * 100
)

print(cell41_percentages.round(2).to_string())

print("\nREGIONAL COVERAGE BY YEAR")

cell41_coverage = (
    cell41_data
    .groupby(["year", "region"], observed=True)
    .size()
    .unstack(fill_value=0)
)

print(cell41_coverage.to_string())

AQUA — YEARLY REGIONAL TARGET DISTRIBUTION AUDIT
Total observations: 12,431
Unparseable dates: 0
Missing region values: 0
Missing target values: 0

OBSERVATION COUNTS BY YEAR AND REGION


Is WQ in range?   No   Yes
year region               
2021 Eluru       190   142
     Nellore     228   110
2022 Eluru       456   437
     Nellore     270  1049
2023 Eluru       316   913
     Nellore     114   748
2024 Eluru       210  1619
     Nellore      56   608
2025 Eluru       243  2761
2026 Eluru       284  1677

TARGET PERCENTAGES WITHIN EACH YEAR-REGION GROUP
Is WQ in range?    No   Yes
year region                
2021 Eluru      57.23 42.77
     Nellore    67.46 32.54
2022 Eluru      51.06 48.94
     Nellore    20.47 79.53
2023 Eluru      25.71 74.29
     Nellore    13.23 86.77
2024 Eluru      11.48 88.52
     Nellore     8.43 91.57
2025 Eluru       8.09 91.91
2026 Eluru      14.48 85.52

REGIONAL COVERAGE BY YEAR
region  Eluru  Nellore
year                  
2021      332      338
2022      893     1319
2023     1229      862
2024     1829      664
2025     3004        0
2026     1961        0


In [37]:
# AQUA — Cell 42
# Name: monthly_regional_target_distribution_audit
# Purpose: Examine monthly target distribution and sampling coverage by region.
# Original dataframe remains unchanged.

cell42_data = df[
    [
        "Date of data collection",
        "region",
        "Is WQ in range?"
    ]
].copy()

cell42_data["collection_date"] = pd.to_datetime(
    cell42_data["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

cell42_data["year_month"] = (
    cell42_data["collection_date"].dt.to_period("M")
)

print("AQUA — MONTHLY REGIONAL TARGET DISTRIBUTION AUDIT")
print(f"Total observations: {len(cell42_data):,}")
print(f"Unparseable dates: {cell42_data['collection_date'].isna().sum():,}")
print(f"Missing region values: {cell42_data['region'].isna().sum():,}")
print(f"Missing target values: {cell42_data['Is WQ in range?'].isna().sum():,}")

# Count target labels within each month and region.
cell42_counts = (
    cell42_data
    .groupby(
        ["year_month", "region", "Is WQ in range?"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
)

# Ensure both target columns are available.
for label in ["No", "Yes"]:
    if label not in cell42_counts.columns:
        cell42_counts[label] = 0

cell42_counts = cell42_counts[["No", "Yes"]]

# Calculate group totals and target percentages.
cell42_summary = cell42_counts.copy()

cell42_summary["Total"] = (
    cell42_summary["No"] + cell42_summary["Yes"]
)

cell42_summary["No (%)"] = (
    cell42_summary["No"] / cell42_summary["Total"] * 100
)

cell42_summary["Yes (%)"] = (
    cell42_summary["Yes"] / cell42_summary["Total"] * 100
)

print("\nMONTHLY TARGET COUNTS AND PERCENTAGES BY REGION")
print(cell42_summary.round(2).to_string())

# Check which months have observations from both regions.
cell42_coverage = (
    cell42_data
    .groupby("year_month", observed=True)["region"]
    .nunique()
)

print("\nMONTHLY REGIONAL COVERAGE")
print(
    cell42_coverage
    .value_counts()
    .sort_index()
    .rename("Number of months")
    .to_string()
)

AQUA — MONTHLY REGIONAL TARGET DISTRIBUTION AUDIT
Total observations: 12,431
Unparseable dates: 0
Missing region values: 0
Missing target values: 0

MONTHLY TARGET COUNTS AND PERCENTAGES BY REGION
Is WQ in range?     No  Yes  Total  No (%)  Yes (%)
year_month region                                  
2021-06    Nellore   0    2      2    0.00   100.00
2021-07    Eluru    12    0     12  100.00     0.00
           Nellore  14   17     31   45.16    54.84
2021-08    Eluru    35   13     48   72.92    27.08
           Nellore  35   16     51   68.63    31.37
2021-09    Eluru    31   23     54   57.41    42.59
           Nellore  48   12     60   80.00    20.00
2021-10    Eluru    13   33     46   28.26    71.74
           Nellore  42    9     51   82.35    17.65
2021-11    Eluru    48   22     70   68.57    31.43
           Nellore  48   24     72   66.67    33.33
2021-12    Eluru    51   51    102   50.00    50.00
           Nellore  41   30     71   57.75    42.25
2022-01    Eluru    28 

In [38]:
# AQUA — Cell 44, Updated cell of 43
# Name: exact_ammonia_column_name_verification
# Purpose: Verify ammonia-related column names directly from the live dataframe.

print("AQUA — LIVE DATAFRAME COLUMN VERIFICATION")
print(f"Total columns: {len(df.columns)}")

print("\nEXACT AMMONIA-RELATED COLUMN NAMES")

ammonia_columns = [
    column
    for column in df.columns
    if "ammonia" in str(column).lower()
    or "tan" in str(column).lower()
    or "nh3" in str(column).lower()
]

if ammonia_columns:
    for column in ammonia_columns:
        print(repr(column))
else:
    print("No ammonia-related columns found.")

print("\nEXACT REQUIRED NON-AMMONIA COLUMN NAMES")

for column in [
    "Date of data collection",
    "region",
    "Is WQ in range?"
]:
    print(f"{column!r}: {column in df.columns}")

AQUA — LIVE DATAFRAME COLUMN VERIFICATION
Total columns: 53

EXACT AMMONIA-RELATED COLUMN NAMES
'Ammonia—TAN (NH3-N) (mg/L)'
'Ammonia—TAN (NH3) (mg/L)'
'Ammonia—NH3 (mg/L)'

EXACT REQUIRED NON-AMMONIA COLUMN NAMES
'Date of data collection': True
'region': True
'Is WQ in range?': True


In [39]:
# AQUA — Cell 45
# RE-EDITED VERSION OF CELL 43
# Name: ammonia_protocol_and_target_distribution_audit
# Purpose: Examine ammonia protocol availability alongside target distribution.
# Original dataframe remains unchanged.

# 1. Verify the complete required schema before accessing columns.

cell45_columns = [
    "Date of data collection",
    "region",
    "Is WQ in range?",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)"
]

cell45_missing_columns = [
    column for column in cell45_columns
    if column not in df.columns
]

print("AQUA — AMMONIA PROTOCOL AND TARGET DISTRIBUTION AUDIT")

if cell45_missing_columns:
    print("\nAUDIT STOPPED — REQUIRED COLUMNS NOT FOUND:")
    for column in cell45_missing_columns:
        print(repr(column))
else:
    # 2. Work on a separate copy; do not modify df.

    cell45_data = df[cell45_columns].copy()

    cell45_data["collection_date"] = pd.to_datetime(
        cell45_data["Date of data collection"],
        format="%m/%d/%Y",
        errors="coerce"
    )

    cell45_data["year_month"] = (
        cell45_data["collection_date"].dt.to_period("M")
    )

    # 3. Check date parsing and required fields.

    print(f"\nTotal observations: {len(cell45_data):,}")
    print(
        "Unparseable dates:",
        cell45_data["collection_date"].isna().sum()
    )
    print(
        "Missing region values:",
        cell45_data["region"].isna().sum()
    )
    print(
        "Missing target values:",
        cell45_data["Is WQ in range?"].isna().sum()
    )

    # 4. Identify protocol availability.

    cell45_ammonia_fields = {
        "Old TAN-N": "Ammonia—TAN (NH3-N) (mg/L)",
        "Transitional TAN": "Ammonia—TAN (NH3) (mg/L)",
        "New NH3": "Ammonia—NH3 (mg/L)"
    }

    print("\nOVERALL AMMONIA FIELD AVAILABILITY")

    for label, column in cell45_ammonia_fields.items():
        count = int(cell45_data[column].notna().sum())
        percentage = count / len(cell45_data) * 100

        print(
            f"{label}: {count:,} records "
            f"({percentage:.2f}%)"
        )

    # 5. Examine monthly distribution from August 2024 onward.

    cell45_recent = cell45_data.loc[
        cell45_data["collection_date"] >= pd.Timestamp("2024-08-01")
    ].copy()

    cell45_recent["old_TAN_N_records"] = (
        cell45_recent["Ammonia—TAN (NH3-N) (mg/L)"].notna()
    )

    cell45_recent["transitional_TAN_records"] = (
        cell45_recent["Ammonia—TAN (NH3) (mg/L)"].notna()
    )

    cell45_recent["new_NH3_records"] = (
        cell45_recent["Ammonia—NH3 (mg/L)"].notna()
    )

    cell45_monthly = (
        cell45_recent
        .groupby("year_month", observed=True)
        .agg(
            observations=("Is WQ in range?", "size"),
            target_no=("Is WQ in range?", lambda x: (x == "No").sum()),
            target_yes=("Is WQ in range?", lambda x: (x == "Yes").sum()),
            old_TAN_N_records=("old_TAN_N_records", "sum"),
            transitional_TAN_records=("transitional_TAN_records", "sum"),
            new_NH3_records=("new_NH3_records", "sum")
        )
    )

    cell45_monthly["target_yes_percent"] = (
        cell45_monthly["target_yes"]
        / cell45_monthly["observations"] * 100
    )

    print("\nMONTHLY TARGET AND AMMONIA AVAILABILITY — AUGUST 2024 ONWARD")
    print(cell45_monthly.round(2).to_string())

    # 6. Examine the same information separately by region.

    cell45_regional = (
        cell45_recent
        .groupby(
            ["year_month", "region"],
            observed=True
        )
        .agg(
            observations=("Is WQ in range?", "size"),
            target_no=("Is WQ in range?", lambda x: (x == "No").sum()),
            target_yes=("Is WQ in range?", lambda x: (x == "Yes").sum()),
            old_TAN_N_records=("old_TAN_N_records", "sum"),
            transitional_TAN_records=("transitional_TAN_records", "sum"),
            new_NH3_records=("new_NH3_records", "sum")
        )
    )

    cell45_regional["target_yes_percent"] = (
        cell45_regional["target_yes"]
        / cell45_regional["observations"] * 100
    )

    print("\nMONTHLY TARGET AND AMMONIA AVAILABILITY BY REGION")
    print(cell45_regional.round(2).to_string())

AQUA — AMMONIA PROTOCOL AND TARGET DISTRIBUTION AUDIT

Total observations: 12,431
Unparseable dates: 0
Missing region values: 0
Missing target values: 0

OVERALL AMMONIA FIELD AVAILABILITY
Old TAN-N: 5,292 records (42.57%)
Transitional TAN: 1,659 records (13.35%)
New NH3: 1,660 records (13.35%)

MONTHLY TARGET AND AMMONIA AVAILABILITY — AUGUST 2024 ONWARD
            observations  target_no  target_yes  old_TAN_N_records  transitional_TAN_records  new_NH3_records  target_yes_percent
year_month                                                                                                                       
2024-08              168         53         115                 85                         0                0               68.45
2024-09              254         36         218                135                         0                0               85.83
2024-10              258         10         248                138                         0                0             

In [40]:
# CELL 46: TARGET DISTRIBUTION BY AMMONIA AVAILABILITY

# 1. Verify required columns before accessing them
required_columns = [
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Is WQ in range?"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise KeyError(f"Required columns not found: {missing_columns}")

# 2. Create temporary availability indicators
ammonia_flags = pd.DataFrame(index=df.index)

ammonia_flags["Old TAN-N"] = df[
    "Ammonia—TAN (NH3-N) (mg/L)"
].notna()

ammonia_flags["Transitional TAN"] = df[
    "Ammonia—TAN (NH3) (mg/L)"
].notna()

ammonia_flags["New NH3"] = df[
    "Ammonia—NH3 (mg/L)"
].notna()

# 3. Classify each record by its exact availability pattern
def classify_ammonia_availability(row):
    available = [
        name for name, present in row.items() if present
    ]

    if not available:
        return "No ammonia measurement"

    return " + ".join(available)

availability_group = ammonia_flags.apply(
    classify_ammonia_availability,
    axis=1
)

# 4. Combine availability groups with the target
analysis_data = pd.DataFrame({
    "Ammonia availability": availability_group,
    "Target": df["Is WQ in range?"]
})

# 5. Calculate target counts and percentages
target_counts = pd.crosstab(
    analysis_data["Ammonia availability"],
    analysis_data["Target"]
)

target_percentages = pd.crosstab(
    analysis_data["Ammonia availability"],
    analysis_data["Target"],
    normalize="index"
).mul(100).round(2)

group_summary = pd.DataFrame(index=target_counts.index)

group_summary["Total records"] = target_counts.sum(axis=1)
group_summary["Target No"] = target_counts.get("No", 0)
group_summary["Target Yes"] = target_counts.get("Yes", 0)
group_summary["Target No (%)"] = target_percentages.get("No", 0)
group_summary["Target Yes (%)"] = target_percentages.get("Yes", 0)

group_summary = group_summary.sort_values(
    "Total records",
    ascending=False
)

# 6. Print one consolidated output
print("=" * 75)
print("TARGET DISTRIBUTION BY EXACT AMMONIA AVAILABILITY")
print("=" * 75)

print("\nAMMONIA AVAILABILITY GROUPS")
print(group_summary.to_string())

print("\n" + "-" * 75)
print("TOTAL RECORDS CHECK")
print("-" * 75)

print(f"Dataset records: {len(df):,}")
print(f"Records classified: {availability_group.notna().sum():,}")
print(f"Records without ammonia measurements: "
      f"{(availability_group == 'No ammonia measurement').sum():,}")

print("\n" + "-" * 75)
print("TARGET COMPLETENESS CHECK")
print("-" * 75)

print(df["Is WQ in range?"].value_counts(dropna=False).to_string())

print("\n" + "=" * 75)
print("END OF CELL 46")
print("=" * 75)

TARGET DISTRIBUTION BY EXACT AMMONIA AVAILABILITY

AMMONIA AVAILABILITY GROUPS
                                        Total records  Target No  Target Yes  Target No (%)  Target Yes (%)
Ammonia availability                                                                                       
No ammonia measurement                           5481        968        4513          17.66           82.34
Old TAN-N                                        5290       1154        4136          21.81           78.19
Transitional TAN + New NH3                       1658        245        1413          14.78           85.22
Old TAN-N + New NH3                                 1          0           1           0.00          100.00
Old TAN-N + Transitional TAN + New NH3              1          0           1           0.00          100.00

---------------------------------------------------------------------------
TOTAL RECORDS CHECK
--------------------------------------------------------------------

In [41]:
# RE-EDITED VERSION OF CELL 47
# Consolidated ammonia value distribution by target

ammonia_columns = {
    "Old TAN-N": "Ammonia—TAN (NH3-N) (mg/L)",
    "Transitional TAN": "Ammonia—TAN (NH3) (mg/L)",
    "New NH3": "Ammonia—NH3 (mg/L)"
}

target_column = "Is WQ in range?"

required = list(ammonia_columns.values()) + [target_column]
missing = [col for col in required if col not in df.columns]

if missing:
    raise KeyError(f"Missing columns: {missing}")

rows = []

for protocol, column in ammonia_columns.items():
    temp = pd.DataFrame({
        "Target": df[target_column],
        "Ammonia value": pd.to_numeric(df[column], errors="coerce")
    }).dropna()

    summary = temp.groupby("Target")["Ammonia value"].agg(
        Count="count",
        Mean="mean",
        Median="median",
        SD="std",
        Minimum="min",
        Maximum="max"
    )

    for target, values in summary.iterrows():
        rows.append({
            "Protocol": protocol,
            "Target": target,
            **values.to_dict()
        })

cell47_result = pd.DataFrame(rows).round(3)

print(cell47_result.to_string(index=False))

        Protocol Target   Count  Mean  Median   SD  Minimum  Maximum
       Old TAN-N     No 1154.00  0.24    0.12 0.35     0.00     3.00
       Old TAN-N    Yes 4138.00  0.15    0.12 0.13     0.00     0.49
Transitional TAN     No  245.00  0.29    0.25 0.33     0.00     2.00
Transitional TAN    Yes 1414.00  0.14    0.25 0.14     0.00     0.50
         New NH3     No  245.00  0.04    0.02 0.05     0.00     0.33
         New NH3    Yes 1415.00  0.01    0.01 0.01     0.00     0.05


In [42]:
# CELL 49: DOCUMENTED AMMONIA THRESHOLD COMPARISON

protocols = {
    "Old TAN-N": {
        "column": "Ammonia—TAN (NH3-N) (mg/L)",
        "threshold": 0.5
    },
    "Transitional TAN": {
        "column": "Ammonia—TAN (NH3) (mg/L)",
        "threshold": None
    },
    "New NH3": {
        "column": "Ammonia—NH3 (mg/L)",
        "threshold": 0.05
    }
}

target_column = "Is WQ in range?"

required = [item["column"] for item in protocols.values()] + [target_column]
missing = [col for col in required if col not in df.columns]

if missing:
    raise KeyError(f"Missing columns: {missing}")

rows = []

for protocol, details in protocols.items():
    column = details["column"]
    threshold = details["threshold"]

    temp = pd.DataFrame({
        "Target": df[target_column],
        "Value": pd.to_numeric(df[column], errors="coerce")
    }).dropna()

    for target in ["No", "Yes"]:
        group = temp[temp["Target"] == target]
        count = len(group)

        if threshold is not None:
            above = int((group["Value"] >= threshold).sum())
            above_pct = round(above / count * 100, 2) if count else None
        else:
            above = None
            above_pct = None

        rows.append({
            "Protocol": protocol,
            "Target": target,
            "Count": count,
            "Median": round(group["Value"].median(), 3) if count else None,
            "Maximum": round(group["Value"].max(), 3) if count else None,
            "Documented threshold": threshold,
            "At or above threshold": above,
            "At or above threshold (%)": above_pct
        })

cell49_result = pd.DataFrame(rows)
print(cell49_result.to_string(index=False))

        Protocol Target  Count  Median  Maximum  Documented threshold  At or above threshold  At or above threshold (%)
       Old TAN-N     No   1154    0.12     3.00                  0.50                 168.00                      14.56
       Old TAN-N    Yes   4138    0.12     0.49                  0.50                   0.00                       0.00
Transitional TAN     No    245    0.25     2.00                   NaN                    NaN                        NaN
Transitional TAN    Yes   1414    0.25     0.50                   NaN                    NaN                        NaN
         New NH3     No    245    0.02     0.33                  0.05                  73.00                      29.80
         New NH3    Yes   1415    0.01     0.05                  0.05                   8.00                       0.57


In [43]:
# CELL 50: NEW NH3 THRESHOLD EXCEPTIONS

column = "Ammonia—NH3 (mg/L)"
target_column = "Is WQ in range?"
threshold = 0.05

required = [
    column,
    target_column,
    "Date of data collection",
    "pond_id",
    "region",
    "Type"
]

missing = [col for col in required if col not in df.columns]

if missing:
    raise KeyError(f"Missing columns: {missing}")

values = pd.to_numeric(df[column], errors="coerce")

exceptions = df.loc[
    (df[target_column] == "Yes") &
    (values >= threshold),
    required
].copy()

exceptions["NH3 value"] = values.loc[exceptions.index]

exceptions = exceptions.drop(columns=[column])

print(exceptions.to_string(index=False))

Is WQ in range? Date of data collection       pond_id region    Type  NH3 value
            Yes              07/31/2026 ara2_8ee1e5e2  Eluru Morning       0.05
            Yes              07/09/2026 ara2_e985ca4a  Eluru Morning       0.05
            Yes              06/17/2026 ara2_7a7eace1  Eluru Morning       0.05
            Yes              04/14/2026 ara2_725e7f3b  Eluru Morning       0.05
            Yes              04/08/2026 ara2_c4b7f174  Eluru Morning       0.05
            Yes              03/26/2026 ara2_e985ca4a  Eluru Morning       0.05
            Yes              03/11/2026 ara2_970ce848  Eluru Morning       0.05
            Yes              02/23/2026 ara2_e985ca4a  Eluru Morning       0.05


In [44]:
# CELL 51: NH3 THRESHOLD PRECISION INVESTIGATION

column = "Ammonia—NH3 (mg/L)"
target_column = "Is WQ in range?"
threshold = 0.05

required = [
    column,
    target_column,
    "Date of data collection",
    "pond_id",
    "region",
    "Type"
]

missing = [col for col in required if col not in df.columns]

if missing:
    raise KeyError(f"Missing columns: {missing}")

values = pd.to_numeric(df[column], errors="coerce")

exceptions = df.loc[
    (df[target_column] == "Yes") &
    (values >= threshold),
    required
].copy()

exceptions["Stored NH3 value"] = values.loc[exceptions.index]
exceptions["Difference from threshold"] = (
    exceptions["Stored NH3 value"] - threshold
)

exceptions["Stored value equals threshold"] = (
    exceptions["Stored NH3 value"] == threshold
)

exceptions = exceptions.drop(columns=[column])

print(exceptions.to_string(index=False))

Is WQ in range? Date of data collection       pond_id region    Type  Stored NH3 value  Difference from threshold  Stored value equals threshold
            Yes              07/31/2026 ara2_8ee1e5e2  Eluru Morning              0.05                       0.00                           True
            Yes              07/09/2026 ara2_e985ca4a  Eluru Morning              0.05                       0.00                           True
            Yes              06/17/2026 ara2_7a7eace1  Eluru Morning              0.05                       0.00                           True
            Yes              04/14/2026 ara2_725e7f3b  Eluru Morning              0.05                       0.00                           True
            Yes              04/08/2026 ara2_c4b7f174  Eluru Morning              0.05                       0.00                           True
            Yes              03/26/2026 ara2_e985ca4a  Eluru Morning              0.05                       0.00                 

In [45]:
# CELL 52: OLD TAN-N THRESHOLD BOUNDARY AUDIT

column = "Ammonia—TAN (NH3-N) (mg/L)"
target_column = "Is WQ in range?"
threshold = 0.50

required = [
    column, target_column,
    "Date of data collection", "pond_id", "region", "Type"
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

values = pd.to_numeric(df[column], errors="coerce")

audit = df.loc[
    values.notna() & (values >= threshold),
    required
].copy()

audit["TAN-N value"] = values.loc[audit.index]

summary = (
    audit.groupby(target_column)["TAN-N value"]
    .agg(Count="count", Minimum="min", Median="median", Maximum="max")
    .round(4)
)

print(summary.to_string())

                 Count  Minimum  Median  Maximum
Is WQ in range?                                 
No                 168     0.50    0.72     3.00


In [46]:
# CELL 53: AMMONIA THRESHOLD EXCEPTION SUMMARY

protocols = {
    "Old TAN-N": ("Ammonia—TAN (NH3-N) (mg/L)", 0.50),
    "New NH3": ("Ammonia—NH3 (mg/L)", 0.05)
}

target_column = "Is WQ in range?"
rows = []

for protocol, (column, threshold) in protocols.items():
    values = pd.to_numeric(df[column], errors="coerce")

    temp = pd.DataFrame({
        "Target": df[target_column],
        "Value": values
    }).dropna()

    for target in ["No", "Yes"]:
        group = temp[temp["Target"] == target]
        count = len(group)
        exceptions = int((group["Value"] >= threshold).sum())

        rows.append({
            "Protocol": protocol,
            "Target": target,
            "Measured records": count,
            "Threshold (mg/L)": threshold,
            "At or above threshold": exceptions,
            "Exception rate (%)": round(
                exceptions / count * 100, 2
            ) if count else None
        })

print(pd.DataFrame(rows).to_string(index=False))

 Protocol Target  Measured records  Threshold (mg/L)  At or above threshold  Exception rate (%)
Old TAN-N     No              1154              0.50                    168               14.56
Old TAN-N    Yes              4138              0.50                      0                0.00
  New NH3     No               245              0.05                     73               29.80
  New NH3    Yes              1415              0.05                      8                0.57


In [47]:
# CELL 54: PREDICTOR CANDIDATE AVAILABILITY AUDIT

candidate_columns = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "TDS (ppt)",
    "Temp (in °C)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Feed amount (kg)",
    "Species",
    "Stocking density",
    "Weather",
    "Equipment",
    "Observer",
    "Pond status",
    "Group",
    "Water color"
]

available = [col for col in candidate_columns if col in df.columns]
not_found = [col for col in candidate_columns if col not in df.columns]

audit = pd.DataFrame({
    "Column": available,
    "Data type": [str(df[col].dtype) for col in available],
    "Available records": [df[col].notna().sum() for col in available],
    "Missing records": [df[col].isna().sum() for col in available],
    "Missing (%)": [
        round(df[col].isna().mean() * 100, 2) for col in available
    ],
    "Unique values": [df[col].nunique(dropna=True) for col in available]
})

print(audit.to_string(index=False))
if not_found:
    print("\nCandidate names not found:", not_found)

                    Column Data type  Available records  Missing records  Missing (%)  Unique values
                 DO (mg/L)   float64              12412               19         0.15            196
                        pH   float64              12412               19         0.15            229
         Turbidity (in cm)   float64               4898             7533        60.60             66
                 TDS (ppt)   float64               4582             7849        63.14           1017
              Temp (in °C)   float64              12410               21         0.17            144
Ammonia—TAN (NH3-N) (mg/L)   float64               5292             7139        57.43            124
  Ammonia—TAN (NH3) (mg/L)   float64               1659            10772        86.65              7
        Ammonia—NH3 (mg/L)   float64               1660            10771        86.65             23
          Feed amount (kg)   float64               6756             5675        45.65      

In [48]:
# CELL 55: STOCKING DENSITY COLUMN VERIFICATION

matches = [
    col for col in df.columns
    if any(term in col.lower() for term in ["dens", "stock", "fish count"])
]

print(pd.DataFrame({
    "Matching column names": matches
}).to_string(index=False))

      Matching column names
Stocking density (per acre)


In [49]:
# CELL 56: CORRECTED PREDICTOR CANDIDATE AVAILABILITY AUDIT

candidate_columns = [
    "DO (mg/L)",
    "pH",
    "Turbidity (in cm)",
    "TDS (ppt)",
    "Temp (in °C)",
    "Ammonia—TAN (NH3-N) (mg/L)",
    "Ammonia—TAN (NH3) (mg/L)",
    "Ammonia—NH3 (mg/L)",
    "Feed amount (kg)",
    "Stocking density (per acre)",
    "Species",
    "Weather",
    "Equipment",
    "Observer",
    "Pond status",
    "Group",
    "Water color"
]

audit = pd.DataFrame({
    "Column": candidate_columns,
    "Available": [df[col].notna().sum() for col in candidate_columns],
    "Missing": [df[col].isna().sum() for col in candidate_columns],
    "Missing (%)": [
        round(df[col].isna().mean() * 100, 2)
        for col in candidate_columns
    ],
    "Unique values": [
        df[col].nunique(dropna=True) for col in candidate_columns
    ]
})

print(audit.to_string(index=False))

                     Column  Available  Missing  Missing (%)  Unique values
                  DO (mg/L)      12412       19         0.15            196
                         pH      12412       19         0.15            229
          Turbidity (in cm)       4898     7533        60.60             66
                  TDS (ppt)       4582     7849        63.14           1017
               Temp (in °C)      12410       21         0.17            144
 Ammonia—TAN (NH3-N) (mg/L)       5292     7139        57.43            124
   Ammonia—TAN (NH3) (mg/L)       1659    10772        86.65              7
         Ammonia—NH3 (mg/L)       1660    10771        86.65             23
           Feed amount (kg)       6756     5675        45.65            168
Stocking density (per acre)       9780     2651        21.33            564
                    Species       8650     3781        30.42             19
                    Weather      11703      728         5.86             36
            

In [50]:
# CELL 57: STOCKING DENSITY NUMERIC QUALITY AUDIT

column = "Stocking density (per acre)"

values = pd.to_numeric(df[column], errors="coerce").dropna()

q1 = values.quantile(0.25)
q3 = values.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

result = pd.DataFrame([{
    "Available records": len(values),
    "Missing records": df[column].isna().sum(),
    "Mean": round(values.mean(), 3),
    "Median": round(values.median(), 3),
    "Std deviation": round(values.std(), 3),
    "Minimum": round(values.min(), 3),
    "Q1": round(q1, 3),
    "Q3": round(q3, 3),
    "Maximum": round(values.max(), 3),
    "Zero values": int((values == 0).sum()),
    "Negative values": int((values < 0).sum()),
    "Lower IQR bound": round(lower_bound, 3),
    "Upper IQR bound": round(upper_bound, 3),
    "IQR outliers": int(((values < lower_bound) | (values > upper_bound)).sum())
}])

print(result.to_string(index=False))

 Available records  Missing records    Mean  Median  Std deviation  Minimum      Q1      Q3   Maximum  Zero values  Negative values  Lower IQR bound  Upper IQR bound  IQR outliers
              9780             2651 8654.24 2791.00       21826.00     0.00 2285.00 5059.00 235294.00            5                0         -1876.00          9220.00          1411


In [51]:
# CELL 58: EXTREME STOCKING DENSITY RECORD INVESTIGATION

column = "Stocking density (per acre)"

values = pd.to_numeric(df[column], errors="coerce")

required = [
    "Date of data collection",
    "pond_id",
    "region",
    "Type",
    column,
    "Species",
    "Pond status"
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

investigation = df.loc[values.nlargest(20).index, required].copy()
investigation[column] = values.loc[investigation.index]

print(investigation.sort_values(column, ascending=False).to_string(index=False))

Date of data collection       pond_id region    Type  Stocking density (per acre)                                  Species Pond status
             05/27/2025 ara2_7bd7da7f  Eluru Evening                    235294.00 Catla (Catla catla), Rohu (Labeo rohita)   Treatment
             05/27/2025 ara2_7bd7da7f  Eluru Morning                    235294.00 Catla (Catla catla), Rohu (Labeo rohita)   Treatment
             07/10/2026 ara2_8cf52ce5  Eluru Evening                    200000.00 Catla (Catla catla), Rohu (Labeo rohita)   Treatment
             07/10/2026 ara2_8cf52ce5  Eluru Morning                    200000.00 Catla (Catla catla), Rohu (Labeo rohita)   Treatment
             06/16/2026 ara2_8cf52ce5  Eluru Morning                    200000.00 Catla (Catla catla), Rohu (Labeo rohita)   Treatment
             05/13/2026 ara2_8cf52ce5  Eluru Evening                    200000.00 Catla (Catla catla), Rohu (Labeo rohita)   Treatment
             05/13/2026 ara2_8cf52ce5  Eluru Morning   

In [52]:
# CELL 59: EXTREME STOCKING DENSITY REPETITION AUDIT

column = "Stocking density (per acre)"

values = pd.to_numeric(df[column], errors="coerce")

extreme = df.loc[
    values >= 100000,
    ["pond_id", "Date of data collection", "Type", column]
].copy()

extreme[column] = values.loc[extreme.index]

result = (
    extreme.groupby(["pond_id", column])
    .agg(
        Records=("Date of data collection", "size"),
        Collection_dates=("Date of data collection", "nunique"),
        Types_recorded=("Type", "nunique"),
        First_date=("Date of data collection", "min"),
        Last_date=("Date of data collection", "max")
    )
    .reset_index()
)

print(result.to_string(index=False))

      pond_id  Stocking density (per acre)  Records  Collection_dates  Types_recorded First_date  Last_date
ara2_2167f891                    133333.00       18                 9               2 01/06/2025 12/23/2024
ara2_27387783                    117647.00        2                 1               2 03/11/2026 03/11/2026
ara2_30c31874                    119048.00        9                 5               2 04/08/2026 08/03/2026
ara2_30c31874                    166667.00        7                 4               2 01/22/2026 12/24/2025
ara2_4441fb4f                    175438.00       32                17               2 01/03/2025 12/19/2024
ara2_6452000e                    100000.00        9                 5               2 09/10/2025 12/12/2025
ara2_6452000e                    140000.00        2                 1               2 05/09/2025 05/09/2025
ara2_6452000e                    180000.00       22                11               2 01/07/2025 12/23/2024
ara2_6ec2f0e5               

In [53]:
# CELL 60: CORRECTED STOCKING DENSITY REPETITION AUDIT

column = "Stocking density (per acre)"

values = pd.to_numeric(df[column], errors="coerce")
dates = pd.to_datetime(
    df["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

temp = pd.DataFrame({
    "pond_id": df["pond_id"],
    "Stocking density": values,
    "Collection date": dates,
    "Type": df["Type"]
})

temp = temp[
    temp["Stocking density"] >= 100000
].copy()

result = (
    temp.groupby(["pond_id", "Stocking density"])
    .agg(
        Records=("Collection date", "size"),
        Collection_dates=("Collection date", "nunique"),
        Types_recorded=("Type", "nunique"),
        First_date=("Collection date", "min"),
        Last_date=("Collection date", "max")
    )
    .reset_index()
)

result["First_date"] = result["First_date"].dt.strftime("%d-%m-%Y")
result["Last_date"] = result["Last_date"].dt.strftime("%d-%m-%Y")

print(result.to_string(index=False))

      pond_id  Stocking density  Records  Collection_dates  Types_recorded First_date  Last_date
ara2_2167f891         133333.00       18                 9               2 16-12-2024 10-04-2025
ara2_27387783         117647.00        2                 1               2 11-03-2026 11-03-2026
ara2_30c31874         119048.00        9                 5               2 08-04-2026 03-08-2026
ara2_30c31874         166667.00        7                 4               2 08-10-2025 22-01-2026
ara2_4441fb4f         175438.00       32                17               2 23-10-2024 08-05-2025
ara2_6452000e         100000.00        9                 5               2 10-09-2025 12-12-2025
ara2_6452000e         140000.00        2                 1               2 09-05-2025 09-05-2025
ara2_6452000e         180000.00       22                11               2 17-10-2024 08-04-2025
ara2_6ec2f0e5         121359.00       15                 8               2 10-12-2025 14-07-2026
ara2_7bd7da7f         117647.0

In [54]:
# CELL 61: FINAL PREDICTOR ELIGIBILITY AUDIT

predictor_review = [
    ("DO (mg/L)", "Candidate", "Direct water-quality measurement"),
    ("pH", "Candidate", "Direct water-quality measurement"),
    ("Temp (in °C)", "Candidate", "Direct water-quality measurement"),
    ("Turbidity (in cm)", "Conditional", "Substantial missingness; verify measurement reliability"),
    ("TDS (ppt)", "Conditional", "Extreme values require reliability verification"),
    ("Ammonia—TAN (NH3-N) (mg/L)", "Conditional", "Old measurement protocol; analyse separately"),
    ("Ammonia—TAN (NH3) (mg/L)", "Conditional", "Transitional protocol; threshold not established"),
    ("Ammonia—NH3 (mg/L)", "Conditional", "New protocol; boundary exceptions identified"),
    ("Stocking density (per acre)", "Conditional", "Extreme repeated values require source verification"),
    ("Feed amount (kg)", "Conditional", "Verify whether available at prediction time"),
    ("Species", "Conditional", "Substantial missingness; verify recording consistency"),
    ("Weather", "Conditional", "Verify measurement timing and availability"),
    ("Water color", "Conditional", "Verify measurement consistency"),
    ("Type", "Candidate", "Morning/evening measurement context"),
    ("Date of data collection", "Candidate", "Temporal context; requires time-aware validation"),
    ("region", "Candidate", "Geographic context; assess regional generalisation"),
    ("Equipment", "Conditional", "Potential measurement-device effect"),
    ("Observer", "Conditional", "Potential observer-related measurement effect"),
    ("Pond status", "Conditional", "Verify status availability and meaning"),
    ("Group", "Conditional", "Verify grouping definition and potential confounding"),
    ("Parameters out of range", "Exclude", "Directly reveals the target"),
    ("Have fish been helped", "Exclude", "Post-intervention information"),
    ("Water quality improved after corrective actions", "Exclude", "Post-intervention outcome"),
    ("Is follow up", "Exclude", "Follow-up indicator; not a baseline predictor"),
]

result = pd.DataFrame(
    predictor_review,
    columns=["Column", "Initial decision", "Reason"]
)

result["Present in dataset"] = result["Column"].isin(df.columns)

print(result.to_string(index=False))

                                         Column Initial decision                                                  Reason  Present in dataset
                                      DO (mg/L)        Candidate                        Direct water-quality measurement                True
                                             pH        Candidate                        Direct water-quality measurement                True
                                   Temp (in °C)        Candidate                        Direct water-quality measurement                True
                              Turbidity (in cm)      Conditional Substantial missingness; verify measurement reliability                True
                                      TDS (ppt)      Conditional         Extreme values require reliability verification                True
                     Ammonia—TAN (NH3-N) (mg/L)      Conditional            Old measurement protocol; analyse separately                True
             

In [55]:
# CELL 62: IDENTIFIER AND TARGET ROLE VERIFICATION

role_review = [
    ("pond_id", "Identifier", "Groups repeated observations; not a primary predictor"),
    ("Date of data collection", "Conditional predictor", "Temporal context; requires time-aware validation"),
    ("Time", "Conditional predictor", "Collection-time context"),
    ("Is WQ in range?", "Target", "Prediction outcome; never use as a predictor"),
]

result = pd.DataFrame(
    role_review,
    columns=["Column", "Role", "Reason"]
)

result["Present in dataset"] = result["Column"].isin(df.columns)

print(result.to_string(index=False))

                 Column                  Role                                                Reason  Present in dataset
                pond_id            Identifier Groups repeated observations; not a primary predictor                True
Date of data collection Conditional predictor      Temporal context; requires time-aware validation                True
                   Time Conditional predictor                               Collection-time context               False
        Is WQ in range?                Target          Prediction outcome; never use as a predictor                True


In [56]:
# CELL 63: COLLECTION TIME COLUMN VERIFICATION

matches = [
    column for column in df.columns
    if "time" in column.lower()
]

print(
    pd.DataFrame({
        "Matching column names": matches
    }).to_string(index=False)
    if matches
    else "No column containing 'time' found."
)

  Matching column names
Time of data collection


In [57]:
# CELL 64: FINAL PREDICTOR ELIGIBILITY REGISTER

decisions = [
    ("DO (mg/L)", "Candidate", "Direct water-quality measurement"),
    ("pH", "Candidate", "Direct water-quality measurement"),
    ("Temp (in °C)", "Candidate", "Direct water-quality measurement"),
    ("Type", "Candidate", "Morning/evening measurement context"),

    ("Date of data collection", "Conditional", "Temporal context; requires time-aware validation"),
    ("Time of data collection", "Conditional", "Collection-time context"),
    ("region", "Conditional", "Geographic context; assess generalisation"),
    ("Turbidity (in cm)", "Conditional", "Substantial missingness"),
    ("TDS (ppt)", "Conditional", "Extreme values require reliability verification"),
    ("Ammonia—TAN (NH3-N) (mg/L)", "Conditional", "Old measurement protocol"),
    ("Ammonia—TAN (NH3) (mg/L)", "Conditional", "Transitional measurement protocol"),
    ("Ammonia—NH3 (mg/L)", "Conditional", "New protocol; boundary exceptions"),
    ("Stocking density (per acre)", "Conditional", "Extreme repeated values require verification"),
    ("Feed amount (kg)", "Conditional", "Prediction-time availability needs verification"),
    ("Species", "Conditional", "Missingness and recording consistency"),
    ("Weather", "Conditional", "Measurement timing needs verification"),
    ("Water color", "Conditional", "Measurement consistency needs verification"),
    ("Equipment", "Conditional", "Potential measurement-device effect"),
    ("Observer", "Conditional", "Potential observer-related effect"),
    ("Pond status", "Conditional", "Meaning and prediction-time availability"),
    ("Group", "Conditional", "Definition and potential confounding"),

    ("pond_id", "Identifier", "Repeated observations; grouping and validation"),
    ("Is WQ in range?", "Target", "Prediction outcome; never a predictor"),

    ("Parameters out of range", "Exclude", "Direct target leakage"),
    ("Have fish been helped", "Exclude", "Post-intervention information"),
    ("Water quality improved after corrective actions", "Exclude", "Post-intervention outcome"),
    ("Is follow up", "Exclude", "Follow-up information"),
]

register = pd.DataFrame(
    decisions,
    columns=["Column", "Decision", "Reason"]
)

register["Present"] = register["Column"].isin(df.columns)

print(register.to_string(index=False))

                                         Column    Decision                                           Reason  Present
                                      DO (mg/L)   Candidate                 Direct water-quality measurement     True
                                             pH   Candidate                 Direct water-quality measurement     True
                                   Temp (in °C)   Candidate                 Direct water-quality measurement     True
                                           Type   Candidate              Morning/evening measurement context     True
                        Date of data collection Conditional Temporal context; requires time-aware validation     True
                        Time of data collection Conditional                          Collection-time context     True
                                         region Conditional        Geographic context; assess generalisation     True
                              Turbidity (in cm) Conditio

In [58]:
# CELL 66: RE-EDITED VERSION OF CELL 65
# CONSOLIDATED DATA QUALITY AUDIT FINDINGS

import os

findings = [
    ("Dataset integrity", "12,431 observations, 53 columns and 317 ponds; no exact duplicate rows.", "Verified"),
    ("Dataset coverage", "June 2021 to August 2026; regional coverage changes over time.", "Verified"),
    ("Missingness", "Missingness varies by parameter; alkalinity and hardness are entirely empty.", "Verified"),
    ("Structural missingness", "Follow-up and intervention fields are often absent by design or workflow.", "Requires careful handling"),
    ("Measurement reliability", "Extreme TDS and stocking-density values require source verification.", "Unresolved"),
    ("Ammonia protocols", "Old TAN-N, transitional TAN and new NH3 measurements must be treated separately.", "Verified"),
    ("Target definition", "The target is imbalanced and documented classification criteria changed over time.", "Requires modelling consideration"),
    ("Target leakage", "Parameters out of range and post-intervention fields must be excluded from predictors.", "Verified"),
    ("Repeated observations", "Ponds have unequal observation counts and repeated measurements.", "Verified"),
    ("Temporal sampling", "Monitoring intervals vary; short intervals are strongly associated with follow-up visits.", "Verified"),
    ("Predictor eligibility", "Candidate predictors require prediction-time and reliability checks.", "Initial review complete"),
    ("Generalisation", "Regional coverage and pond composition change over time.", "Requires validation strategy"),
]

audit_summary = pd.DataFrame(
    findings,
    columns=["Audit area", "Finding", "Status"]
)

output_path = "../outputs/tables/consolidated_data_quality_audit_findings.csv"

os.makedirs("../outputs/tables", exist_ok=True)
audit_summary.to_csv(output_path, index=False)

print(audit_summary.to_string(index=False))

             Audit area                                                                                   Finding                           Status
      Dataset integrity                   12,431 observations, 53 columns and 317 ponds; no exact duplicate rows.                         Verified
       Dataset coverage                            June 2021 to August 2026; regional coverage changes over time.                         Verified
            Missingness              Missingness varies by parameter; alkalinity and hardness are entirely empty.                         Verified
 Structural missingness                 Follow-up and intervention fields are often absent by design or workflow.        Requires careful handling
Measurement reliability                      Extreme TDS and stocking-density values require source verification.                       Unresolved
      Ammonia protocols          Old TAN-N, transitional TAN and new NH3 measurements must be treated separately.     

##  Final Audit Conclusion

The dataset passed the initial structural integrity checks, with 12,431 observations across 317 ponds and 53 variables.

The audit identified important considerations involving structural missingness, measurement reliability, ammonia protocol changes, target definition, repeated pond-level observations, temporal sampling, and potential target leakage.

These findings will guide the exploratory analysis, statistical methodology, predictor selection, and validation strategy.

The original dataset has been preserved without modifying or deleting observations. Any subsequent cleaning, transformation, exclusion, or feature engineering will be documented separately.

**Audit outcome:** Suitable for further exploratory and statistical investigation, subject to the limitations identified in this notebook.